In [5]:
# ============================================================================
# SECTION 0: SETUP & CONFIGURATION
# ============================================================================

# Standard library imports
import os
import gc
import multiprocessing as mp
import sys
import json
import time
import logging
import warnings
import random
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Union, Any
import pickle

# Data manipulation
import numpy as np
import pandas as pd
import datatable as dt
from datatable import f, by

# Machine Learning
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler

# Deep Learning
import torch

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

# Progress tracking
from tqdm.auto import tqdm
import progressbar

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# ============================================================================
# Global Configuration
# ============================================================================

# Core Parameters
WINDOW_SIZE = 20                    # Sequence window around phosphorylation site
RANDOM_SEED = 42                    # For reproducibility
EXPERIMENT_NAME = "exp_3"           # Experiment identifier
BASE_DIR = f"results/{EXPERIMENT_NAME}"
MAX_SEQUENCE_LENGTH = 5000          # Filter long sequences
BALANCE_CLASSES = True              # 1:1 positive:negative ratio
USE_DATATABLE = True                # Use datatable for speed optimization
BATCH_SIZE = 32                     # For transformer training
GRADIENT_ACCUMULATION_STEPS = 2     # Memory optimization
USE_MIXED_PRECISION = True          # For transformer efficiency

# Set all random seeds for reproducibility
def set_all_seeds(seed: int):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_all_seeds(RANDOM_SEED)

# ============================================================================
# Progress Tracking System
# ============================================================================

class ProgressTracker:
    """Comprehensive progress tracking with checkpoint management"""
    
    def __init__(self, exp_dir: str, auto_cleanup: bool = True):
        self.exp_dir = exp_dir
        self.auto_cleanup = auto_cleanup
        self.progress_file = os.path.join(exp_dir, 'progress_tracker.json')
        self.start_time = datetime.now()
        
        # Create directory structure
        self._create_directories()
        
        # Load or initialize progress
        self.progress = self._load_progress()
        
        # Memory monitoring
        self.memory_threshold = 0.8  # 80% memory usage triggers cleanup
        
    def _create_directories(self):
        """Create all required directories"""
        directories = [
            self.exp_dir,
            os.path.join(self.exp_dir, 'checkpoints'),
            os.path.join(self.exp_dir, 'checkpoints/data_preprocessing'),
            os.path.join(self.exp_dir, 'checkpoints/feature_extraction'),
            os.path.join(self.exp_dir, 'checkpoints/ml_models'),
            os.path.join(self.exp_dir, 'checkpoints/transformers'),
            os.path.join(self.exp_dir, 'checkpoints/ensemble'),
            os.path.join(self.exp_dir, 'ml_models'),
            os.path.join(self.exp_dir, 'transformers'),
            os.path.join(self.exp_dir, 'ensemble'),
            os.path.join(self.exp_dir, 'final_report'),
            os.path.join(self.exp_dir, 'logs'),
            os.path.join(self.exp_dir, 'plots'),
            os.path.join(self.exp_dir, 'plots/data_exploration'),
            os.path.join(self.exp_dir, 'plots/feature_analysis'),
            os.path.join(self.exp_dir, 'plots/ml_models'),
            os.path.join(self.exp_dir, 'plots/transformers'),
            os.path.join(self.exp_dir, 'plots/ensemble'),
            os.path.join(self.exp_dir, 'plots/error_analysis'),
            os.path.join(self.exp_dir, 'plots/final_evaluation'),
            os.path.join(self.exp_dir, 'plots/final_report'),
            os.path.join(self.exp_dir, 'tables'),
            os.path.join(self.exp_dir, 'models')
        ]
        
        for directory in directories:
            os.makedirs(directory, exist_ok=True)
    
    def _load_progress(self) -> Dict:
        """Load progress from file if exists"""
        if os.path.exists(self.progress_file):
            with open(self.progress_file, 'r') as f:
                return json.load(f)
        else:
            return {
                'experiment_start': self.start_time.isoformat(),
                'completed_steps': {},
                'checkpoints': {},
                'metadata': {
                    'experiment_name': EXPERIMENT_NAME,
                    'random_seed': RANDOM_SEED,
                    'window_size': WINDOW_SIZE
                }
            }
    
    def _save_progress(self):
        """Save progress to file"""
        with open(self.progress_file, 'w') as f:
            json.dump(self.progress, f, indent=2, default=str)
    
    def mark_completed(self, step_name: str, metadata: Dict = None, checkpoint_data: Any = None):
        """Mark a step as completed and optionally save checkpoint"""
        completion_time = datetime.now()
        self.progress['completed_steps'][step_name] = {
            'completed_at': completion_time.isoformat(),
            'duration_seconds': (completion_time - self.start_time).total_seconds(),
            'metadata': metadata or {}
        }
        
        if checkpoint_data is not None:
            checkpoint_path = os.path.join(
                self.exp_dir, 'checkpoints', f'{step_name.replace(" ", "_").lower()}.pkl'
            )
            with open(checkpoint_path, 'wb') as f:
                pickle.dump(checkpoint_data, f, protocol=4)
            self.progress['checkpoints'][step_name] = checkpoint_path
        
        self._save_progress()
        
        # Check memory and cleanup if needed
        if self.auto_cleanup:
            self._check_memory_usage()
    
    def is_completed(self, step_name: str) -> bool:
        """Check if a step is already completed"""
        return step_name in self.progress['completed_steps']
    
    def resume_from_checkpoint(self, step_name: str) -> Any:
        """Resume from a checkpoint if exists"""
        if step_name in self.progress['checkpoints']:
            checkpoint_path = self.progress['checkpoints'][step_name]
            if os.path.exists(checkpoint_path):
                with open(checkpoint_path, 'rb') as f:
                    return pickle.load(f)
        return None
    
    def get_progress_summary(self) -> Dict:
        """Get summary of progress"""
        total_steps = 10  # Total number of major sections
        completed_steps = len(self.progress['completed_steps'])
        
        return {
            'total_steps': total_steps,
            'completed_steps': completed_steps,
            'percentage': (completed_steps / total_steps) * 100,
            'elapsed_time': str(datetime.now() - self.start_time),
            'completed': list(self.progress['completed_steps'].keys())
        }
    
    def get_memory_usage(self) -> Dict:
        """Get current memory usage"""
        try:
            import psutil
            process = psutil.Process(os.getpid())
            memory_info = process.memory_info()
            return {
                'rss_mb': memory_info.rss / (1024 * 1024),
                'vms_mb': memory_info.vms / (1024 * 1024),
                'percent': process.memory_percent()
            }
        except ImportError:
            return {'rss_mb': 0, 'vms_mb': 0, 'percent': 0}
    
    def _check_memory_usage(self):
        """Check memory usage and trigger cleanup if needed"""
        memory = self.get_memory_usage()
        if memory['percent'] > self.memory_threshold * 100:
            self.trigger_cleanup()
    
    def trigger_cleanup(self):
        """Trigger memory cleanup"""
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    def force_retrain(self, step_name: str):
        """Force retrain by removing a completed step"""
        if step_name in self.progress['completed_steps']:
            del self.progress['completed_steps'][step_name]
        if step_name in self.progress['checkpoints']:
            checkpoint_path = self.progress['checkpoints'][step_name]
            if os.path.exists(checkpoint_path):
                os.remove(checkpoint_path)
            del self.progress['checkpoints'][step_name]
        self._save_progress()
    
    def export_progress_report(self) -> str:
        """Export detailed progress report"""
        report = f"""
Phosphorylation Prediction Experiment Progress Report
=====================================================
Experiment: {EXPERIMENT_NAME}
Started: {self.progress['experiment_start']}
Current Time: {datetime.now().isoformat()}
Elapsed: {datetime.now() - self.start_time}

Progress Summary:
-----------------
"""
        summary = self.get_progress_summary()
        report += f"Completed: {summary['completed_steps']}/{summary['total_steps']} steps ({summary['percentage']:.1f}%)\n\n"
        
        report += "Completed Steps:\n"
        for step, info in self.progress['completed_steps'].items():
            report += f"- {step}: {info['completed_at']} (Duration: {info['duration_seconds']:.1f}s)\n"
        
        report += f"\nMemory Usage:\n"
        memory = self.get_memory_usage()
        report += f"- RSS: {memory['rss_mb']:.1f} MB\n"
        report += f"- VMS: {memory['vms_mb']:.1f} MB\n"
        report += f"- Percent: {memory['percent']:.1f}%\n"
        
        return report

# ============================================================================
# Logging Setup
# ============================================================================

def setup_logging(log_dir: str):
    """Setup comprehensive logging"""
    log_file = os.path.join(log_dir, 'experiment.log')
    
    # Configure logging
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler(sys.stdout)
        ]
    )
    
    logger = logging.getLogger(__name__)
    logger.info("="*80)
    logger.info(f"Phosphorylation Prediction Experiment: {EXPERIMENT_NAME}")
    logger.info(f"Started at: {datetime.now()}")
    logger.info("="*80)
    
    return logger


# ============================================================================
# Environment Information
# ============================================================================

def log_environment_info(logger):
    """Log complete environment information"""
    logger.info("\nEnvironment Information:")
    logger.info(f"Python version: {sys.version}")
    logger.info(f"NumPy version: {np.__version__}")
    logger.info(f"Pandas version: {pd.__version__}")
    logger.info(f"PyTorch version: {torch.__version__}")
    
    # GPU information
    if torch.cuda.is_available():
        logger.info(f"CUDA available: Yes")
        logger.info(f"CUDA version: {torch.version.cuda}")
        logger.info(f"GPU count: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            logger.info(f"GPU {i}: {torch.cuda.get_device_name(i)}")
            logger.info(f"GPU {i} Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
    else:
        logger.info("CUDA available: No (CPU mode)")
    
    # Memory information
    try:
        import psutil
        memory = psutil.virtual_memory()
        logger.info(f"Total RAM: {memory.total / 1e9:.1f} GB")
        logger.info(f"Available RAM: {memory.available / 1e9:.1f} GB")
    except ImportError:
        logger.info("psutil not available for memory information")


# ============================================================================
# Configuration Export
# ============================================================================

def export_configuration(exp_dir: str):
    """Export complete experiment configuration"""
    config = {
        'experiment': {
            'name': EXPERIMENT_NAME,
            'base_dir': BASE_DIR,
            'created_at': datetime.now().isoformat()
        },
        'data': {
            'window_size': WINDOW_SIZE,
            'max_sequence_length': MAX_SEQUENCE_LENGTH,
            'balance_classes': BALANCE_CLASSES,
            'use_datatable': USE_DATATABLE
        },
        'training': {
            'random_seed': RANDOM_SEED,
            'batch_size': BATCH_SIZE,
            'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
            'use_mixed_precision': USE_MIXED_PRECISION
        },
        'environment': {
            'python_version': sys.version,
            'numpy_version': np.__version__,
            'pandas_version': pd.__version__,
            'torch_version': torch.__version__,
            'cuda_available': torch.cuda.is_available(),
            'gpu_count': torch.cuda.device_count() if torch.cuda.is_available() else 0
        }
    }
    
    config_file = os.path.join(exp_dir, 'experiment_config.yaml')
    with open(config_file, 'w') as f:
        json.dump(config, f, indent=2, default=str)
    
    return config


# ============================================================================
# Initialize Everything
# ============================================================================

print("Initializing Phosphorylation Prediction Experiment...")
print(f"Experiment Name: {EXPERIMENT_NAME}")
print(f"Base Directory: {BASE_DIR}")

# Initialize progress tracker
progress_tracker = ProgressTracker(BASE_DIR)

# Setup logging
logger = setup_logging(os.path.join(BASE_DIR, 'logs'))

# Log environment information
log_environment_info(logger)

# Export configuration
config = export_configuration(BASE_DIR)
logger.info(f"Configuration exported to: {os.path.join(BASE_DIR, 'experiment_config.yaml')}")

# Display progress summary
summary = progress_tracker.get_progress_summary()
print(f"\nProgress: {summary['completed_steps']}/{summary['total_steps']} steps completed ({summary['percentage']:.1f}%)")
if summary['completed_steps'] > 0:
    print("Completed steps:", ", ".join(summary['completed']))

print("\nSetup completed successfully!")
print("="*80)

Initializing Phosphorylation Prediction Experiment...
Experiment Name: exp_3
Base Directory: results/exp_3
2025-07-21 12:47:45,740 - __main__ - INFO - ================================================================================
2025-07-21 12:47:45,741 - __main__ - INFO - Phosphorylation Prediction Experiment: exp_3
2025-07-21 12:47:45,741 - __main__ - INFO - Started at: 2025-07-21 12:47:45.741531
2025-07-21 12:47:45,742 - __main__ - INFO - ================================================================================
2025-07-21 12:47:45,742 - __main__ - INFO - 
Environment Information:
2025-07-21 12:47:45,742 - __main__ - INFO - Python version: 3.9.21 (main, Dec 11 2024, 16:35:24) [MSC v.1929 64 bit (AMD64)]
2025-07-21 12:47:45,743 - __main__ - INFO - NumPy version: 1.26.4
2025-07-21 12:47:45,743 - __main__ - INFO - Pandas version: 2.2.3
2025-07-21 12:47:45,744 - __main__ - INFO - PyTorch version: 2.5.1+cu121
2025-07-21 12:47:45,744 - __main__ - INFO - CUDA available: Yes
2025-07

In [14]:
# ============================================================================
# IMPORTS
# ============================================================================
import os
import gc
import json
import pickle
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader as TorchDataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR
from torch.cuda.amp import autocast, GradScaler

# Transformers
from transformers import AutoTokenizer, AutoModel

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

# Progress tracking
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    """Configuration for the orthogonal ensemble"""
    # Experiment settings
    EXPERIMENT_NAME = "orthogonal_ensemble_6_components"
    RANDOM_SEED = 42
    BASE_DIR = "results/exp_3"
    
    # Model settings
    WINDOW_SIZE = 20  # ±10 AA around phosphorylation site
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Feature dimensions (updated based on actual data)
    AAC_DIM = 20
    DPC_DIM = 400
    TPC_DIM = 7996  # Fixed: actual dimension from your data
    PHYS_DIM = 656
    BINARY_DIM = 820
    
    # Transformation methods from Section 4
    SELECTED_CONFIGS = {
        'aac': {
            'method': 'polynomial',
            'description': 'Polynomial interactions (degree=2)',
            'model': 'xgboost'
        },
        'dpc': {
            'method': 'pca_30',
            'description': 'PCA with 30 components',
            'model': 'catboost'
        },
        'tpc': {
            'method': 'pca_50',
            'description': 'PCA with 50 components',
            'model': 'catboost'
        },
        'physicochemical': {
            'method': 'mutual_info_500',
            'description': 'Mutual Information (top 500)',
            'model': 'catboost'
        },
        'binary': {
            'method': 'variance_pca_200',
            'description': 'Variance threshold + PCA (200 components)',
            'model': 'xgboost'
        }
    }
    
    # Training settings
    PHASE1_EPOCHS = 50
    PHASE2_EPOCHS = 30
    PHASE3_EPOCHS = 100
    
    # Batch sizes (adjusted for 8GB GPU)
    PHASE1_BATCH_SIZE = 256
    PHASE2_BATCH_SIZE = 512
    PHASE3_BATCH_SIZE = 128
    
    # Learning rates
    PHASE1_LR = 1e-3
    PHASE2_LR = 5e-4
    PHASE3_LR = {
        'aac_net': 1e-5,
        'dpc_net': 1e-5,
        'tpc_net': 5e-6,
        'phys_net': 1e-5,
        'binary_net': 1e-5,
        'transformer': 2e-6,
        'fusion': 1e-4
    }
    
    # Regularization
    DROPOUT_RATES = {
        'aac': 0.1,
        'dpc': 0.2,
        'tpc': 0.3,
        'phys': 0.2,
        'binary': 0.2,
        'transformer': 0.3,
        'fusion': 0.1
    }
    
    # Early stopping
    EARLY_STOPPING_PATIENCE = 10
    
    # Transformer settings
    TRANSFORMER_MODEL = "facebook/esm2_t6_8M_UR50D"
    MAX_LENGTH = 64

# ============================================================================
# COMPONENT 1: AAC NETWORK
# ============================================================================
class AACNetwork(nn.Module):
    """Neural network for Amino Acid Composition features"""
    def __init__(self, input_dim=210, dropout_rate=0.1):  # Updated for polynomial features
        super(AACNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

# ============================================================================
# COMPONENT 2: DPC NETWORK
# ============================================================================
class DPCNetwork(nn.Module):
    """Neural network for Dipeptide Composition features"""
    def __init__(self, input_dim=30, dropout_rate=0.2):  # PCA-30
        super(DPCNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

# ============================================================================
# COMPONENT 3: TPC NETWORK
# ============================================================================
class TPCNetwork(nn.Module):
    """Neural network for Tripeptide Composition features"""
    def __init__(self, input_dim=50, dropout_rate=0.3):  # PCA-50
        super(TPCNetwork, self).__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

# ============================================================================
# COMPONENT 4: PHYSICOCHEMICAL NETWORK
# ============================================================================
class PhysicochemicalNetwork(nn.Module):
    """Neural network for Physicochemical features"""
    def __init__(self, input_dim=500, dropout_rate=0.2):  # Mutual Info 500
        super(PhysicochemicalNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

# ============================================================================
# COMPONENT 5: BINARY NETWORK
# ============================================================================
class BinaryNetwork(nn.Module):
    """Neural network for Binary encoding features"""
    def __init__(self, input_dim=200, dropout_rate=0.2):  # Variance + PCA-200
        super(BinaryNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

# ============================================================================
# COMPONENT 6: SEQUENCE TRANSFORMER
# ============================================================================
class SequenceTransformer(nn.Module):
    """ESM-2 based transformer for sequence understanding"""
    def __init__(self, dropout_rate=0.3):
        super(SequenceTransformer, self).__init__()
        
        # Load pre-trained ESM-2 model
        self.esm_model = AutoModel.from_pretrained(Config.TRANSFORMER_MODEL)
        
        # Freeze ESM-2 embeddings initially
        for param in self.esm_model.embeddings.parameters():
            param.requires_grad = False
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(320, 128),  # ESM-2 t6 has 320 hidden dim
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, input_ids, attention_mask):
        # Get ESM-2 outputs
        outputs = self.esm_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        
        # Use the representation at the center position (phosphorylation site)
        hidden_states = outputs.last_hidden_state
        
        # Get center position representation
        batch_size = hidden_states.size(0)
        center_idx = hidden_states.size(1) // 2
        center_repr = hidden_states[:, center_idx, :]
        
        # Classify
        return self.classifier(center_repr)

# ============================================================================
# FUSION NETWORK WITH ATTENTION
# ============================================================================
class FusionNetwork(nn.Module):
    """Fusion network with attention mechanism to combine all components"""
    def __init__(self, dropout_rate=0.1):
        super(FusionNetwork, self).__init__()
        
        # Component prediction encoder
        self.prediction_encoder = nn.Sequential(
            nn.Linear(6, 32),
            nn.LayerNorm(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        
        # Attention mechanism for component weighting
        self.attention = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 6),
            nn.Softmax(dim=1)  # Component weights sum to 1
        )
        
        # Final prediction layers
        self.predictor = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )
    
    def forward(self, component_predictions):
        """
        Args:
            component_predictions: tensor of shape (batch_size, 6)
                                 containing predictions from all 6 components
        Returns:
            final_prediction: tensor of shape (batch_size, 1)
            attention_weights: tensor of shape (batch_size, 6)
        """
        # Encode component predictions
        encoded = self.prediction_encoder(component_predictions)
        
        # Calculate attention weights
        attention_weights = self.attention(encoded)
        
        # Apply attention to original predictions
        weighted_preds = attention_weights * component_predictions
        
        # Re-encode weighted predictions
        weighted_encoded = self.prediction_encoder(weighted_preds)
        
        # Final prediction
        final_pred = self.predictor(weighted_encoded)
        
        return final_pred, attention_weights

# ============================================================================
# COMPLETE ORTHOGONAL ENSEMBLE
# ============================================================================
class OrthogonalEnsemble(nn.Module):
    """Complete 6-component orthogonal ensemble"""
    def __init__(self):
        super(OrthogonalEnsemble, self).__init__()
        
        # Initialize all components with correct dimensions from Section 4
        self.aac_net = AACNetwork(input_dim=210, dropout_rate=Config.DROPOUT_RATES['aac'])  # Polynomial features
        self.dpc_net = DPCNetwork(input_dim=30, dropout_rate=Config.DROPOUT_RATES['dpc'])  # PCA-30
        self.tpc_net = TPCNetwork(input_dim=50, dropout_rate=Config.DROPOUT_RATES['tpc'])  # PCA-50
        self.phys_net = PhysicochemicalNetwork(input_dim=500, dropout_rate=Config.DROPOUT_RATES['phys'])  # MI-500
        self.binary_net = BinaryNetwork(input_dim=200, dropout_rate=Config.DROPOUT_RATES['binary'])  # Var+PCA-200
        self.transformer = SequenceTransformer(Config.DROPOUT_RATES['transformer'])
        self.fusion_net = FusionNetwork(Config.DROPOUT_RATES['fusion'])
        
        # Component names for easy iteration
        self.component_names = ['aac', 'dpc', 'tpc', 'phys', 'binary', 'transformer']
        
    def forward(self, aac_features, dpc_features, tpc_features, phys_features, 
                binary_features, input_ids, attention_mask):
        """
        Forward pass through all components and fusion
        """
        # Get predictions from each component
        aac_pred = self.aac_net(aac_features)
        dpc_pred = self.dpc_net(dpc_features)
        tpc_pred = self.tpc_net(tpc_features)
        phys_pred = self.phys_net(phys_features)
        binary_pred = self.binary_net(binary_features)
        transformer_pred = self.transformer(input_ids, attention_mask)
        
        # Stack all predictions
        component_predictions = torch.cat([
            aac_pred, dpc_pred, tpc_pred, phys_pred, binary_pred, transformer_pred
        ], dim=1)
        
        # Fusion
        final_pred, attention_weights = self.fusion_net(component_predictions)
        
        return {
            'final_prediction': final_pred,
            'component_predictions': component_predictions,
            'attention_weights': attention_weights,
            'individual_predictions': {
                'aac': aac_pred,
                'dpc': dpc_pred,
                'tpc': tpc_pred,
                'phys': phys_pred,
                'binary': binary_pred,
                'transformer': transformer_pred
            }
        }
    
    def freeze_components(self):
        """Freeze all component models (for Phase 2 training)"""
        for name in self.component_names:
            if name == 'transformer':
                for param in self.transformer.parameters():
                    param.requires_grad = False
            else:
                component = getattr(self, f"{name}_net")
                for param in component.parameters():
                    param.requires_grad = False
    
    def unfreeze_components(self):
        """Unfreeze all component models (for Phase 3 training)"""
        for name in self.component_names:
            if name == 'transformer':
                # Keep embeddings frozen, unfreeze rest
                for param in self.transformer.parameters():
                    param.requires_grad = True
                for param in self.transformer.esm_model.embeddings.parameters():
                    param.requires_grad = False
            else:
                component = getattr(self, f"{name}_net")
                for param in component.parameters():
                    param.requires_grad = True

# ============================================================================
# DATA LOADING AND PREPROCESSING
# ============================================================================
class DataLoader:
    """Load and preprocess data from checkpoints"""
    def __init__(self, progress_tracker, config=Config):
        self.progress_tracker = progress_tracker
        self.config = config
        self.data = {}
        
    def load_all_data(self):
        """Load all required data from checkpoints"""
        print("\n" + "="*80)
        print("LOADING DATA FROM CHECKPOINTS")
        print("="*80)
        
        # Load from Section 1 (data_loading)
        print("\nLoading from Section 1 checkpoint...")
        data_checkpoint = self.progress_tracker.resume_from_checkpoint("data_loading")
        if not data_checkpoint:
            raise RuntimeError("Section 1 checkpoint not found! Please run Section 1 first.")
        
        self.data['df_final'] = data_checkpoint['df_final']
        # Fix: Use correct column names (capital letters)
        self.data['sequences'] = self.data['df_final']['Sequence'].values
        self.data['positions'] = self.data['df_final']['Position'].values
        print(f"✓ Loaded {len(self.data['df_final'])} samples")
        print(f"  Columns in df_final: {list(self.data['df_final'].columns)}")
        
        # Load from Section 2 (feature_extraction)
        print("\nLoading from Section 2 checkpoint...")
        feature_checkpoint = self.progress_tracker.resume_from_checkpoint("feature_extraction")
        if not feature_checkpoint:
            raise RuntimeError("Section 2 checkpoint not found! Please run Section 2 first.")
        
        feature_matrices = feature_checkpoint['feature_matrices']
        self.data['aac_features'] = feature_matrices['aac']
        self.data['dpc_features'] = feature_matrices['dpc']
        self.data['tpc_features'] = feature_matrices['tpc']
        self.data['physicochemical_features'] = feature_matrices['physicochemical']
        self.data['binary_features'] = feature_matrices['binary']
        
        # Get target from df_final since it's the most reliable source
        self.data['target'] = self.data['df_final']['target'].values
        
        print(f"✓ Loaded feature matrices:")
        print(f"  - AAC: {self.data['aac_features'].shape}")
        print(f"  - DPC: {self.data['dpc_features'].shape}")
        print(f"  - TPC: {self.data['tpc_features'].shape}")
        print(f"  - Physicochemical: {self.data['physicochemical_features'].shape}")
        print(f"  - Binary: {self.data['binary_features'].shape}")
        
        # Load from Section 3 (data_splitting)
        print("\nLoading from Section 3 checkpoint...")
        split_checkpoint = self.progress_tracker.resume_from_checkpoint("data_splitting")
        if not split_checkpoint:
            raise RuntimeError("Section 3 checkpoint not found! Please run Section 3 first.")
        
        # Handle different checkpoint structures
        if 'splits' in split_checkpoint:
            self.data['train_indices'] = split_checkpoint['splits']['train_indices']
            self.data['test_indices'] = split_checkpoint['splits']['test_indices']
        else:
            # Fallback for older checkpoint format
            self.data['train_indices'] = split_checkpoint.get('train_indices')
            self.data['test_indices'] = split_checkpoint.get('test_indices')
        
        if self.data['train_indices'] is None or self.data['test_indices'] is None:
            raise RuntimeError("Train/test indices not found in Section 3 checkpoint!")
        
        print(f"✓ Loaded splits:")
        print(f"  - Train: {len(self.data['train_indices'])} samples")
        print(f"  - Test: {len(self.data['test_indices'])} samples")
        
        return self.data
    
    def extract_sequence_windows(self):
        """Extract sequence windows around phosphorylation sites"""
        print("\nExtracting sequence windows...")
        
        window_sequences = []
        
        for seq, pos in zip(self.data['sequences'], self.data['positions']):
            # Get window around phosphorylation site
            start = max(0, pos - self.config.WINDOW_SIZE // 2)
            end = min(len(seq), pos + self.config.WINDOW_SIZE // 2 + 1)
            
            window = seq[start:end]
            
            # Pad if necessary
            if len(window) < self.config.WINDOW_SIZE + 1:
                if start == 0:
                    window = 'X' * (self.config.WINDOW_SIZE + 1 - len(window)) + window
                else:
                    window = window + 'X' * (self.config.WINDOW_SIZE + 1 - len(window))
            
            window_sequences.append(window)
        
        self.data['window_sequences'] = window_sequences
        print(f"✓ Extracted {len(window_sequences)} sequence windows")
        
        return window_sequences
    
    def prepare_features(self):
        """Prepare and transform features using Section 4 methods"""
        print("\nPreparing features with Section 4 transformations...")
        
        # Convert to numpy arrays if they're DataFrames
        for feature_type in ['aac', 'dpc', 'tpc', 'physicochemical', 'binary']:
            feature_key = f"{feature_type}_features"
            if isinstance(self.data[feature_key], pd.DataFrame):
                numeric_cols = self.data[feature_key].select_dtypes(include=[np.number]).columns
                self.data[feature_key] = self.data[feature_key][numeric_cols].values
        
        # Import necessary libraries
        from sklearn.decomposition import PCA
        from sklearn.preprocessing import PolynomialFeatures
        from sklearn.feature_selection import SelectKBest, mutual_info_classif, VarianceThreshold
        
        # Initialize storage
        self.scalers = {}
        self.transformers = {}
        
        # Get training data for fitting transformers
        train_indices = self.data['train_indices']
        
        # ========== AAC: Polynomial Features ==========
        print("  Processing AAC features (Polynomial interactions)...")
        poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
        aac_train = self.data['aac_features'][train_indices]
        poly.fit(aac_train)
        self.data['aac_scaled'] = poly.transform(self.data['aac_features'])
        self.transformers['aac'] = poly
        print(f"    AAC: {self.data['aac_features'].shape[1]} → {self.data['aac_scaled'].shape[1]} features")
        
        # ========== DPC: PCA with 30 components ==========
        print("  Processing DPC features (PCA-30)...")
        dpc_scaler = StandardScaler()
        dpc_train = self.data['dpc_features'][train_indices]
        dpc_scaler.fit(dpc_train)
        dpc_scaled = dpc_scaler.transform(self.data['dpc_features'])
        
        dpc_pca = PCA(n_components=30, random_state=Config.RANDOM_SEED)
        dpc_pca.fit(dpc_scaled[train_indices])
        self.data['dpc_scaled'] = dpc_pca.transform(dpc_scaled)
        self.scalers['dpc'] = dpc_scaler
        self.transformers['dpc'] = dpc_pca
        print(f"    DPC: {self.data['dpc_features'].shape[1]} → {self.data['dpc_scaled'].shape[1]} features")
        print(f"    Variance explained: {dpc_pca.explained_variance_ratio_.sum():.3f}")
        
        # ========== TPC: PCA with 50 components ==========
        print("  Processing TPC features (PCA-50)...")
        tpc_scaler = StandardScaler()
        tpc_train = self.data['tpc_features'][train_indices]
        tpc_scaler.fit(tpc_train)
        tpc_scaled = tpc_scaler.transform(self.data['tpc_features'])
        
        tpc_pca = PCA(n_components=50, random_state=Config.RANDOM_SEED)
        tpc_pca.fit(tpc_scaled[train_indices])
        self.data['tpc_scaled'] = tpc_pca.transform(tpc_scaled)
        self.scalers['tpc'] = tpc_scaler
        self.transformers['tpc'] = tpc_pca
        print(f"    TPC: {self.data['tpc_features'].shape[1]} → {self.data['tpc_scaled'].shape[1]} features")
        print(f"    Variance explained: {tpc_pca.explained_variance_ratio_.sum():.3f}")
        
        # ========== Physicochemical: Mutual Information Selection (500 features) ==========
        print("  Processing Physicochemical features (Mutual Info 500)...")
        phys_selector = SelectKBest(score_func=mutual_info_classif, k=500)
        phys_train = self.data['physicochemical_features'][train_indices]
        y_train = self.data['target'][train_indices]
        phys_selector.fit(phys_train, y_train)
        self.data['physicochemical_scaled'] = phys_selector.transform(self.data['physicochemical_features'])
        self.transformers['physicochemical'] = phys_selector
        print(f"    Physicochemical: {self.data['physicochemical_features'].shape[1]} → {self.data['physicochemical_scaled'].shape[1]} features")
        
        # ========== Binary: Variance Threshold + PCA (200 components) ==========
        print("  Processing Binary features (Variance threshold + PCA-200)...")
        # Step 1: Variance threshold
        var_selector = VarianceThreshold(threshold=0.01)
        binary_train = self.data['binary_features'][train_indices]
        var_selector.fit(binary_train)
        binary_var_selected = var_selector.transform(self.data['binary_features'])
        
        # Step 2: Standardize
        binary_scaler = StandardScaler()
        binary_scaler.fit(binary_var_selected[train_indices])
        binary_scaled = binary_scaler.transform(binary_var_selected)
        
        # Step 3: PCA
        binary_pca = PCA(n_components=200, random_state=Config.RANDOM_SEED)
        binary_pca.fit(binary_scaled[train_indices])
        self.data['binary_scaled'] = binary_pca.transform(binary_scaled)
        
        self.transformers['binary'] = {
            'var_selector': var_selector,
            'scaler': binary_scaler,
            'pca': binary_pca
        }
        print(f"    Binary: {self.data['binary_features'].shape[1]} → {binary_var_selected.shape[1]} (variance) → {self.data['binary_scaled'].shape[1]} features")
        print(f"    Variance explained: {binary_pca.explained_variance_ratio_.sum():.3f}")
        
        print("✓ Features prepared and transformed using Section 4 methods")
        
    def tokenize_sequences(self):
        """Tokenize sequences for transformer"""
        print("\nTokenizing sequences...")
        
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(Config.TRANSFORMER_MODEL)
        
        # Tokenize all sequences
        tokenized = tokenizer(
            self.data['window_sequences'],
            padding='max_length',
            truncation=True,
            max_length=Config.MAX_LENGTH,
            return_tensors='pt'
        )
        
        self.data['input_ids'] = tokenized['input_ids']
        self.data['attention_mask'] = tokenized['attention_mask']
        
        print(f"✓ Tokenized sequences: {self.data['input_ids'].shape}")
        
        return tokenized

# ============================================================================
# DATASET CLASS
# ============================================================================
class PhosphoDataset(Dataset):
    """Dataset for phosphorylation prediction"""
    def __init__(self, indices, data_dict):
        self.indices = indices
        self.data = data_dict
        
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        # Get actual index
        actual_idx = self.indices[idx]
        
        # Get all features
        sample = {
            'aac': torch.tensor(self.data['aac_scaled'][actual_idx], dtype=torch.float32),
            'dpc': torch.tensor(self.data['dpc_scaled'][actual_idx], dtype=torch.float32),
            'tpc': torch.tensor(self.data['tpc_scaled'][actual_idx], dtype=torch.float32),
            'phys': torch.tensor(self.data['physicochemical_scaled'][actual_idx], dtype=torch.float32),
            'binary': torch.tensor(self.data['binary_scaled'][actual_idx], dtype=torch.float32),
            'input_ids': self.data['input_ids'][actual_idx],
            'attention_mask': self.data['attention_mask'][actual_idx],
            'label': torch.tensor(self.data['target'][actual_idx], dtype=torch.float32)
        }
        
        return sample

# ============================================================================
# LOSS FUNCTIONS
# ============================================================================
class MultiObjectiveLoss(nn.Module):
    """Multi-objective loss for end-to-end training"""
    def __init__(self, diversity_weight=0.1, component_weight=0.05):
        super(MultiObjectiveLoss, self).__init__()
        self.bce_loss = nn.BCELoss()
        self.diversity_weight = diversity_weight
        self.component_weight = component_weight
        
    def forward(self, outputs, labels):
        """
        Calculate multi-objective loss
        """
        # Main prediction loss
        main_loss = self.bce_loss(outputs['final_prediction'].squeeze(), labels)
        
        # Component diversity loss (encourage using multiple components)
        attention_weights = outputs['attention_weights']
        max_attention = torch.max(attention_weights, dim=1)[0]
        diversity_loss = torch.mean(max_attention)  # Minimize max attention
        
        # Component performance loss (individual components should be good)
        component_losses = []
        for pred in outputs['individual_predictions'].values():
            component_losses.append(self.bce_loss(pred.squeeze(), labels))
        avg_component_loss = torch.mean(torch.stack(component_losses))
        
        # Combined loss
        total_loss = (main_loss + 
                     self.diversity_weight * diversity_loss + 
                     self.component_weight * avg_component_loss)
        
        return {
            'total_loss': total_loss,
            'main_loss': main_loss,
            'diversity_loss': diversity_loss,
            'component_loss': avg_component_loss
        }

# ============================================================================
# TRAINING FUNCTIONS
# ============================================================================
class Trainer:
    """Training manager for the orthogonal ensemble"""
    def __init__(self, model, data_loader, config=Config):
        self.model = model
        self.data_loader = data_loader
        self.config = config
        self.device = config.DEVICE
        self.results = defaultdict(list)
        
    def calculate_metrics(self, y_true, y_pred, y_prob):
        """Calculate evaluation metrics"""
        return {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'auc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.0
        }
    
    def train_individual_component(self, component_name, component_model, train_loader, val_loader, epochs=50):
        """Train an individual component (Phase 1)"""
        print(f"\n{'='*60}")
        print(f"Training {component_name.upper()} Component")
        print(f"{'='*60}")
        
        # Move model to device
        component_model = component_model.to(self.device)
        
        # Setup optimizer and scheduler
        optimizer = AdamW(component_model.parameters(), lr=Config.PHASE1_LR, weight_decay=1e-4)
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
        criterion = nn.BCELoss()
        
        # Training metrics
        best_val_f1 = 0
        patience = 0
        train_losses = []
        val_losses = []
        val_f1_scores = []
        
        for epoch in range(epochs):
            # Training phase
            component_model.train()
            train_loss = 0
            all_preds = []
            all_labels = []
            
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
            
            for batch in progress_bar:
                # Get appropriate features based on component
                if component_name == 'transformer':
                    inputs = batch['input_ids'].to(self.device)
                    attention_mask = batch['attention_mask'].to(self.device)
                    outputs = component_model(inputs, attention_mask)
                else:
                    feature_key = component_name.replace('_net', '')
                    inputs = batch[feature_key].to(self.device)
                    outputs = component_model(inputs)
                
                labels = batch['label'].to(self.device)
                
                loss = criterion(outputs.squeeze(), labels)
                
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(component_model.parameters(), 1.0)
                optimizer.step()
                
                train_loss += loss.item()
                all_preds.extend(outputs.detach().cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
                progress_bar.set_postfix({'loss': loss.item()})
            
            # Calculate training metrics
            train_loss /= len(train_loader)
            train_preds_binary = (np.array(all_preds) > 0.5).astype(int).flatten()
            train_metrics = self.calculate_metrics(all_labels, train_preds_binary, np.array(all_preds).flatten())
            
            # Validation phase
            component_model.eval()
            val_loss = 0
            val_preds = []
            val_labels = []
            
            with torch.no_grad():
                for batch in val_loader:
                    if component_name == 'transformer':
                        inputs = batch['input_ids'].to(self.device)
                        attention_mask = batch['attention_mask'].to(self.device)
                        outputs = component_model(inputs, attention_mask)
                    else:
                        feature_key = component_name.replace('_net', '')
                        inputs = batch[feature_key].to(self.device)
                        outputs = component_model(inputs)
                    
                    labels = batch['label'].to(self.device)
                    loss = criterion(outputs.squeeze(), labels)
                    
                    val_loss += loss.item()
                    val_preds.extend(outputs.cpu().numpy())
                    val_labels.extend(labels.cpu().numpy())
            
            # Calculate validation metrics
            val_loss /= len(val_loader)
            val_preds_binary = (np.array(val_preds) > 0.5).astype(int).flatten()
            val_metrics = self.calculate_metrics(val_labels, val_preds_binary, np.array(val_preds).flatten())
            
            # Update scheduler
            scheduler.step()
            
            # Store metrics
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            val_f1_scores.append(val_metrics['f1'])
            
            # Print epoch summary
            print(f"\nEpoch {epoch+1}/{epochs}")
            print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            print(f"Train F1: {train_metrics['f1']:.4f} | Val F1: {val_metrics['f1']:.4f}")
            print(f"Val Metrics - Acc: {val_metrics['accuracy']:.4f}, Prec: {val_metrics['precision']:.4f}, Rec: {val_metrics['recall']:.4f}, AUC: {val_metrics['auc']:.4f}")
            
            # Early stopping
            if val_metrics['f1'] > best_val_f1:
                best_val_f1 = val_metrics['f1']
                patience = 0
                # Save best model
                best_model_state = component_model.state_dict()
            else:
                patience += 1
                if patience >= Config.EARLY_STOPPING_PATIENCE:
                    print(f"\nEarly stopping triggered after {epoch+1} epochs")
                    break
        
        # Load best model
        component_model.load_state_dict(best_model_state)
        
        # Store results
        self.results[f'{component_name}_train_losses'] = train_losses
        self.results[f'{component_name}_val_losses'] = val_losses
        self.results[f'{component_name}_val_f1_scores'] = val_f1_scores
        self.results[f'{component_name}_best_f1'] = best_val_f1
        
        print(f"\n✓ {component_name} training complete. Best Val F1: {best_val_f1:.4f}")
        
        return component_model, best_val_f1
    
    def phase1_training(self, train_dataset, val_dataset):
        """Phase 1: Train individual components"""
        print("\n" + "="*80)
        print("PHASE 1: INDIVIDUAL COMPONENT TRAINING")
        print("="*80)
        
        # Create data loaders
        train_loader = TorchDataLoader(train_dataset, batch_size=Config.PHASE1_BATCH_SIZE, shuffle=True)
        val_loader = TorchDataLoader(val_dataset, batch_size=Config.PHASE1_BATCH_SIZE, shuffle=False)
        
        # Train each component
        component_results = {}
        
        # Train AAC Network
        aac_model, aac_f1 = self.train_individual_component(
            'aac', self.model.aac_net, train_loader, val_loader, Config.PHASE1_EPOCHS
        )
        component_results['aac'] = {'model': aac_model, 'f1': aac_f1}
        
        # Train DPC Network
        dpc_model, dpc_f1 = self.train_individual_component(
            'dpc', self.model.dpc_net, train_loader, val_loader, Config.PHASE1_EPOCHS
        )
        component_results['dpc'] = {'model': dpc_model, 'f1': dpc_f1}
        
        # Train TPC Network
        tpc_model, tpc_f1 = self.train_individual_component(
            'tpc', self.model.tpc_net, train_loader, val_loader, Config.PHASE1_EPOCHS
        )
        component_results['tpc'] = {'model': tpc_model, 'f1': tpc_f1}
        
        # Train Physicochemical Network
        phys_model, phys_f1 = self.train_individual_component(
            'phys', self.model.phys_net, train_loader, val_loader, Config.PHASE1_EPOCHS
        )
        component_results['phys'] = {'model': phys_model, 'f1': phys_f1}
        
        # Train Binary Network
        binary_model, binary_f1 = self.train_individual_component(
            'binary', self.model.binary_net, train_loader, val_loader, Config.PHASE1_EPOCHS
        )
        component_results['binary'] = {'model': binary_model, 'f1': binary_f1}
        
        # Train Transformer
        transformer_model, transformer_f1 = self.train_individual_component(
            'transformer', self.model.transformer, train_loader, val_loader, Config.PHASE1_EPOCHS
        )
        component_results['transformer'] = {'model': transformer_model, 'f1': transformer_f1}
        
        # Summary
        print("\n" + "="*60)
        print("PHASE 1 SUMMARY")
        print("="*60)
        for comp_name, comp_data in component_results.items():
            print(f"{comp_name.upper()}: F1 = {comp_data['f1']:.4f}")
        
        self.component_results = component_results
        return component_results
    
    def get_component_predictions(self, data_loader, freeze=True):
        """Get predictions from all components"""
        self.model.eval()
        
        all_component_preds = defaultdict(list)
        all_labels = []
        
        with torch.no_grad():
            for batch in data_loader:
                # Get predictions from each component
                aac_pred = self.model.aac_net(batch['aac'].to(self.device))
                dpc_pred = self.model.dpc_net(batch['dpc'].to(self.device))
                tpc_pred = self.model.tpc_net(batch['tpc'].to(self.device))
                phys_pred = self.model.phys_net(batch['phys'].to(self.device))
                binary_pred = self.model.binary_net(batch['binary'].to(self.device))
                transformer_pred = self.model.transformer(
                    batch['input_ids'].to(self.device),
                    batch['attention_mask'].to(self.device)
                )
                
                all_component_preds['aac'].extend(aac_pred.cpu().numpy())
                all_component_preds['dpc'].extend(dpc_pred.cpu().numpy())
                all_component_preds['tpc'].extend(tpc_pred.cpu().numpy())
                all_component_preds['phys'].extend(phys_pred.cpu().numpy())
                all_component_preds['binary'].extend(binary_pred.cpu().numpy())
                all_component_preds['transformer'].extend(transformer_pred.cpu().numpy())
                
                all_labels.extend(batch['label'].numpy())
        
        # Convert to numpy arrays
        for comp in all_component_preds:
            all_component_preds[comp] = np.array(all_component_preds[comp]).flatten()
        
        return all_component_preds, np.array(all_labels)
    
    def phase2_training(self, train_dataset, val_dataset):
        """Phase 2: Train fusion network"""
        print("\n" + "="*80)
        print("PHASE 2: FUSION NETWORK TRAINING")
        print("="*80)
        
        # Freeze all components
        self.model.freeze_components()
        print("✓ All component models frozen")
        
        # Create data loaders
        train_loader = TorchDataLoader(train_dataset, batch_size=Config.PHASE2_BATCH_SIZE, shuffle=True)
        val_loader = TorchDataLoader(val_dataset, batch_size=Config.PHASE2_BATCH_SIZE, shuffle=False)
        
        # Move model to device
        self.model = self.model.to(self.device)
        
        # Setup optimizer for fusion network only
        fusion_params = list(self.model.fusion_net.parameters())
        optimizer = AdamW(fusion_params, lr=Config.PHASE2_LR, weight_decay=1e-4)
        scheduler = StepLR(optimizer, step_size=10, gamma=0.5)
        criterion = nn.BCELoss()
        
        # Training metrics
        best_val_f1 = 0
        patience = 0
        train_losses = []
        val_losses = []
        val_f1_scores = []
        attention_history = []
        
        for epoch in range(Config.PHASE2_EPOCHS):
            # Training phase
            self.model.train()
            self.model.freeze_components()  # Ensure components stay frozen
            train_loss = 0
            all_preds = []
            all_labels = []
            epoch_attention_weights = []
            
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.PHASE2_EPOCHS}")
            
            for batch in progress_bar:
                # Forward pass through entire model
                outputs = self.model(
                    batch['aac'].to(self.device),
                    batch['dpc'].to(self.device),
                    batch['tpc'].to(self.device),
                    batch['phys'].to(self.device),
                    batch['binary'].to(self.device),
                    batch['input_ids'].to(self.device),
                    batch['attention_mask'].to(self.device)
                )
                
                labels = batch['label'].to(self.device)
                loss = criterion(outputs['final_prediction'].squeeze(), labels)
                
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(fusion_params, 1.0)
                optimizer.step()
                
                train_loss += loss.item()
                all_preds.extend(outputs['final_prediction'].detach().cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                epoch_attention_weights.extend(outputs['attention_weights'].detach().cpu().numpy())
                
                progress_bar.set_postfix({'loss': loss.item()})
            
            # Calculate training metrics
            train_loss /= len(train_loader)
            train_preds_binary = (np.array(all_preds) > 0.5).astype(int).flatten()
            train_metrics = self.calculate_metrics(all_labels, train_preds_binary, np.array(all_preds).flatten())
            
            # Store attention weights
            attention_history.append(np.mean(epoch_attention_weights, axis=0))
            
            # Validation phase
            self.model.eval()
            val_loss = 0
            val_preds = []
            val_labels = []
            val_attention_weights = []
            
            with torch.no_grad():
                for batch in val_loader:
                    outputs = self.model(
                        batch['aac'].to(self.device),
                        batch['dpc'].to(self.device),
                        batch['tpc'].to(self.device),
                        batch['phys'].to(self.device),
                        batch['binary'].to(self.device),
                        batch['input_ids'].to(self.device),
                        batch['attention_mask'].to(self.device)
                    )
                    
                    labels = batch['label'].to(self.device)
                    loss = criterion(outputs['final_prediction'].squeeze(), labels)
                    
                    val_loss += loss.item()
                    val_preds.extend(outputs['final_prediction'].cpu().numpy())
                    val_labels.extend(labels.cpu().numpy())
                    val_attention_weights.extend(outputs['attention_weights'].cpu().numpy())
            
            # Calculate validation metrics
            val_loss /= len(val_loader)
            val_preds_binary = (np.array(val_preds) > 0.5).astype(int).flatten()
            val_metrics = self.calculate_metrics(val_labels, val_preds_binary, np.array(val_preds).flatten())
            
            # Update scheduler
            scheduler.step()
            
            # Store metrics
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            val_f1_scores.append(val_metrics['f1'])
            
            # Print epoch summary
            print(f"\nEpoch {epoch+1}/{Config.PHASE2_EPOCHS}")
            print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            print(f"Train F1: {train_metrics['f1']:.4f} | Val F1: {val_metrics['f1']:.4f}")
            print(f"Val Metrics - Acc: {val_metrics['accuracy']:.4f}, Prec: {val_metrics['precision']:.4f}, Rec: {val_metrics['recall']:.4f}, AUC: {val_metrics['auc']:.4f}")
            
            # Print average attention weights
            avg_attention = np.mean(val_attention_weights, axis=0)
            print(f"Attention Weights - AAC: {avg_attention[0]:.3f}, DPC: {avg_attention[1]:.3f}, TPC: {avg_attention[2]:.3f}, Phys: {avg_attention[3]:.3f}, Binary: {avg_attention[4]:.3f}, Transformer: {avg_attention[5]:.3f}")
            
            # Early stopping
            if val_metrics['f1'] > best_val_f1:
                best_val_f1 = val_metrics['f1']
                patience = 0
                # Save best model
                best_model_state = self.model.state_dict()
            else:
                patience += 1
                if patience >= Config.EARLY_STOPPING_PATIENCE:
                    print(f"\nEarly stopping triggered after {epoch+1} epochs")
                    break
        
        # Load best model
        self.model.load_state_dict(best_model_state)
        
        # Store results
        self.results['fusion_train_losses'] = train_losses
        self.results['fusion_val_losses'] = val_losses
        self.results['fusion_val_f1_scores'] = val_f1_scores
        self.results['fusion_best_f1'] = best_val_f1
        self.results['attention_history'] = attention_history
        
        print(f"\n✓ Fusion network training complete. Best Val F1: {best_val_f1:.4f}")
        
        return best_val_f1
    
    def phase3_training(self, train_dataset, val_dataset):
        """Phase 3: End-to-end fine-tuning"""
        print("\n" + "="*80)
        print("PHASE 3: END-TO-END FINE-TUNING")
        print("="*80)
        
        # Unfreeze all components
        self.model.unfreeze_components()
        print("✓ All component models unfrozen (except ESM-2 embeddings)")
        
        # Create data loaders with smaller batch size
        train_loader = TorchDataLoader(train_dataset, batch_size=Config.PHASE3_BATCH_SIZE, shuffle=True)
        val_loader = TorchDataLoader(val_dataset, batch_size=Config.PHASE3_BATCH_SIZE, shuffle=False)
        
        # Setup optimizers with different learning rates
        param_groups = [
            {'params': self.model.aac_net.parameters(), 'lr': Config.PHASE3_LR['aac_net']},
            {'params': self.model.dpc_net.parameters(), 'lr': Config.PHASE3_LR['dpc_net']},
            {'params': self.model.tpc_net.parameters(), 'lr': Config.PHASE3_LR['tpc_net']},
            {'params': self.model.phys_net.parameters(), 'lr': Config.PHASE3_LR['phys_net']},
            {'params': self.model.binary_net.parameters(), 'lr': Config.PHASE3_LR['binary_net']},
            {'params': self.model.transformer.parameters(), 'lr': Config.PHASE3_LR['transformer']},
            {'params': self.model.fusion_net.parameters(), 'lr': Config.PHASE3_LR['fusion']}
        ]
        
        optimizer = AdamW(param_groups, weight_decay=1e-4)
        scheduler = CosineAnnealingLR(optimizer, T_max=Config.PHASE3_EPOCHS)
        
        # Multi-objective loss
        criterion = MultiObjectiveLoss(diversity_weight=0.1, component_weight=0.05)
        
        # Enable mixed precision training for memory efficiency
        scaler = GradScaler()
        
        # Training metrics
        best_val_f1 = 0
        patience = 0
        train_losses = []
        val_losses = []
        val_f1_scores = []
        component_f1_history = defaultdict(list)
        
        for epoch in range(Config.PHASE3_EPOCHS):
            # Training phase
            self.model.train()
            train_loss = 0
            train_main_loss = 0
            train_diversity_loss = 0
            train_component_loss = 0
            all_preds = []
            all_labels = []
            
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.PHASE3_EPOCHS}")
            
            for batch in progress_bar:
                # Mixed precision training
                with autocast():
                    outputs = self.model(
                        batch['aac'].to(self.device),
                        batch['dpc'].to(self.device),
                        batch['tpc'].to(self.device),
                        batch['phys'].to(self.device),
                        batch['binary'].to(self.device),
                        batch['input_ids'].to(self.device),
                        batch['attention_mask'].to(self.device)
                    )
                    
                labels = batch['label'].to(self.device)
                loss_dict = criterion(outputs, labels)
                loss = loss_dict['total_loss']
                
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                
                train_loss += loss.item()
                train_main_loss += loss_dict['main_loss'].item()
                train_diversity_loss += loss_dict['diversity_loss'].item()
                train_component_loss += loss_dict['component_loss'].item()
                
                all_preds.extend(outputs['final_prediction'].detach().cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
                progress_bar.set_postfix({
                    'loss': loss.item(),
                    'main': loss_dict['main_loss'].item(),
                    'div': loss_dict['diversity_loss'].item()
                })
            
            # Calculate training metrics
            train_loss /= len(train_loader)
            train_main_loss /= len(train_loader)
            train_diversity_loss /= len(train_loader)
            train_component_loss /= len(train_loader)
            
            train_preds_binary = (np.array(all_preds) > 0.5).astype(int).flatten()
            train_metrics = self.calculate_metrics(all_labels, train_preds_binary, np.array(all_preds).flatten())
            
            # Validation phase
            self.model.eval()
            val_loss = 0
            val_preds = []
            val_labels = []
            val_component_preds = defaultdict(list)
            
            with torch.no_grad():
                for batch in val_loader:
                    outputs = self.model(
                        batch['aac'].to(self.device),
                        batch['dpc'].to(self.device),
                        batch['tpc'].to(self.device),
                        batch['phys'].to(self.device),
                        batch['binary'].to(self.device),
                        batch['input_ids'].to(self.device),
                        batch['attention_mask'].to(self.device)
                    )
                    
                    labels = batch['label'].to(self.device)
                    loss_dict = criterion(outputs, labels)
                    
                    val_loss += loss_dict['total_loss'].item()
                    val_preds.extend(outputs['final_prediction'].cpu().numpy())
                    val_labels.extend(labels.cpu().numpy())
                    
                    # Store component predictions
                    for comp_name, comp_pred in outputs['individual_predictions'].items():
                        val_component_preds[comp_name].extend(comp_pred.cpu().numpy())
            
            # Calculate validation metrics
            val_loss /= len(val_loader)
            val_preds_binary = (np.array(val_preds) > 0.5).astype(int).flatten()
            val_metrics = self.calculate_metrics(val_labels, val_preds_binary, np.array(val_preds).flatten())
            
            # Calculate component metrics
            for comp_name, comp_preds in val_component_preds.items():
                comp_preds_binary = (np.array(comp_preds) > 0.5).astype(int).flatten()
                comp_metrics = self.calculate_metrics(val_labels, comp_preds_binary, np.array(comp_preds).flatten())
                component_f1_history[comp_name].append(comp_metrics['f1'])
            
            # Update scheduler
            scheduler.step()
            
            # Store metrics
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            val_f1_scores.append(val_metrics['f1'])
            
            # Print epoch summary
            print(f"\nEpoch {epoch+1}/{Config.PHASE3_EPOCHS}")
            print(f"Train Loss: {train_loss:.4f} (Main: {train_main_loss:.4f}, Div: {train_diversity_loss:.4f}, Comp: {train_component_loss:.4f})")
            print(f"Val Loss: {val_loss:.4f}")
            print(f"Train F1: {train_metrics['f1']:.4f} | Val F1: {val_metrics['f1']:.4f}")
            print(f"Val Metrics - Acc: {val_metrics['accuracy']:.4f}, Prec: {val_metrics['precision']:.4f}, Rec: {val_metrics['recall']:.4f}, AUC: {val_metrics['auc']:.4f}")
            
            # Print component F1 scores
            print("Component F1 Scores:")
            for comp_name in ['aac', 'dpc', 'tpc', 'phys', 'binary', 'transformer']:
                print(f"  {comp_name}: {component_f1_history[comp_name][-1]:.4f}")
            
            # Early stopping
            if val_metrics['f1'] > best_val_f1:
                best_val_f1 = val_metrics['f1']
                patience = 0
                # Save best model
                best_model_state = self.model.state_dict()
                best_epoch = epoch + 1
            else:
                patience += 1
                if patience >= Config.EARLY_STOPPING_PATIENCE:
                    print(f"\nEarly stopping triggered after {epoch+1} epochs")
                    break
        
        # Load best model
        self.model.load_state_dict(best_model_state)
        
        # Store results
        self.results['e2e_train_losses'] = train_losses
        self.results['e2e_val_losses'] = val_losses
        self.results['e2e_val_f1_scores'] = val_f1_scores
        self.results['e2e_best_f1'] = best_val_f1
        self.results['e2e_best_epoch'] = best_epoch
        self.results['component_f1_history'] = dict(component_f1_history)
        
        print(f"\n✓ End-to-end training complete. Best Val F1: {best_val_f1:.4f} (Epoch {best_epoch})")
        
        return best_val_f1

# ============================================================================
# EVALUATION AND VISUALIZATION
# ============================================================================
class Evaluator:
    """Evaluation and visualization tools"""
    def __init__(self, model, device):
        self.model = model
        self.device = device
    
    def evaluate_on_test(self, test_loader):
        """Evaluate model on test set"""
        print("\n" + "="*80)
        print("FINAL EVALUATION ON TEST SET")
        print("="*80)
        
        self.model.eval()
        all_preds = []
        all_labels = []
        all_attention_weights = []
        component_preds = defaultdict(list)
        
        with torch.no_grad():
            for batch in tqdm(test_loader, desc="Evaluating"):
                outputs = self.model(
                    batch['aac'].to(self.device),
                    batch['dpc'].to(self.device),
                    batch['tpc'].to(self.device),
                    batch['phys'].to(self.device),
                    batch['binary'].to(self.device),
                    batch['input_ids'].to(self.device),
                    batch['attention_mask'].to(self.device)
                )
                
                all_preds.extend(outputs['final_prediction'].cpu().numpy())
                all_labels.extend(batch['label'].numpy())
                all_attention_weights.extend(outputs['attention_weights'].cpu().numpy())
                
                for comp_name, comp_pred in outputs['individual_predictions'].items():
                    component_preds[comp_name].extend(comp_pred.cpu().numpy())
        
        # Convert to arrays
        all_preds = np.array(all_preds).flatten()
        all_labels = np.array(all_labels)
        all_attention_weights = np.array(all_attention_weights)
        
        # Calculate metrics
        preds_binary = (all_preds > 0.5).astype(int)
        final_metrics = self.calculate_metrics(all_labels, preds_binary, all_preds)
        
        print("\nFINAL TEST RESULTS:")
        print(f"Accuracy: {final_metrics['accuracy']:.4f}")
        print(f"Precision: {final_metrics['precision']:.4f}")
        print(f"Recall: {final_metrics['recall']:.4f}")
        print(f"F1 Score: {final_metrics['f1']:.4f}")
        print(f"AUC-ROC: {final_metrics['auc']:.4f}")
        
        # Component performance
        print("\nCOMPONENT PERFORMANCE:")
        component_metrics = {}
        for comp_name, comp_preds_list in component_preds.items():
            comp_preds_array = np.array(comp_preds_list).flatten()
            comp_preds_binary = (comp_preds_array > 0.5).astype(int)
            comp_metrics = self.calculate_metrics(all_labels, comp_preds_binary, comp_preds_array)
            component_metrics[comp_name] = comp_metrics
            print(f"{comp_name}: F1 = {comp_metrics['f1']:.4f}, AUC = {comp_metrics['auc']:.4f}")
        
        # Average attention weights
        avg_attention = np.mean(all_attention_weights, axis=0)
        print("\nAVERAGE ATTENTION WEIGHTS:")
        comp_names = ['AAC', 'DPC', 'TPC', 'Phys', 'Binary', 'Transformer']
        for i, name in enumerate(comp_names):
            print(f"{name}: {avg_attention[i]:.3f}")
        
        return {
            'final_metrics': final_metrics,
            'component_metrics': component_metrics,
            'predictions': all_preds,
            'labels': all_labels,
            'attention_weights': all_attention_weights
        }
    
    def calculate_metrics(self, y_true, y_pred, y_prob):
        """Calculate evaluation metrics"""
        return {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'auc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.0
        }
    
    def plot_training_curves(self, results, save_path):
        """Plot training curves for all phases"""
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle('Orthogonal Ensemble Training Progress', fontsize=16)
        
        # Phase 1: Component training
        ax = axes[0, 0]
        for comp in ['aac', 'dpc', 'tpc', 'phys', 'binary', 'transformer']:
            if f'{comp}_val_f1_scores' in results:
                ax.plot(results[f'{comp}_val_f1_scores'], label=comp.upper())
        ax.set_title('Phase 1: Component F1 Scores')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Validation F1')
        ax.legend()
        ax.grid(True)
        
        # Phase 2: Fusion training
        ax = axes[0, 1]
        ax.plot(results['fusion_train_losses'], label='Train Loss')
        ax.plot(results['fusion_val_losses'], label='Val Loss')
        ax.set_title('Phase 2: Fusion Network Loss')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.legend()
        ax.grid(True)
        
        ax = axes[0, 2]
        ax.plot(results['fusion_val_f1_scores'])
        ax.set_title('Phase 2: Fusion F1 Score')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Validation F1')
        ax.grid(True)
        
        # Phase 3: End-to-end training
        ax = axes[1, 0]
        ax.plot(results['e2e_train_losses'], label='Train Loss')
        ax.plot(results['e2e_val_losses'], label='Val Loss')
        ax.set_title('Phase 3: End-to-End Loss')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.legend()
        ax.grid(True)
        
        ax = axes[1, 1]
        ax.plot(results['e2e_val_f1_scores'])
        ax.set_title('Phase 3: End-to-End F1 Score')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Validation F1')
        ax.axhline(y=results['e2e_best_f1'], color='r', linestyle='--', label=f'Best F1: {results["e2e_best_f1"]:.4f}')
        ax.legend()
        ax.grid(True)
        
        # Component F1 evolution during Phase 3
        ax = axes[1, 2]
        for comp in ['aac', 'dpc', 'tpc', 'phys', 'binary', 'transformer']:
            if comp in results['component_f1_history']:
                ax.plot(results['component_f1_history'][comp], label=comp.upper())
        ax.set_title('Phase 3: Component F1 Evolution')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Validation F1')
        ax.legend()
        ax.grid(True)
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        
    def plot_attention_analysis(self, attention_weights, save_path):
        """Plot attention weight analysis"""
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        # Average attention weights
        avg_attention = np.mean(attention_weights, axis=0)
        comp_names = ['AAC', 'DPC', 'TPC', 'Phys', 'Binary', 'Transformer']
        
        ax = axes[0]
        bars = ax.bar(comp_names, avg_attention)
        ax.set_title('Average Attention Weights by Component')
        ax.set_ylabel('Attention Weight')
        ax.set_ylim(0, 1)
        
        # Color bars based on weight
        for i, bar in enumerate(bars):
            if avg_attention[i] == max(avg_attention):
                bar.set_color('darkgreen')
            elif avg_attention[i] == min(avg_attention):
                bar.set_color('darkred')
            else:
                bar.set_color('steelblue')
        
        # Attention weight distribution
        ax = axes[1]
        ax.boxplot([attention_weights[:, i] for i in range(6)], labels=comp_names)
        ax.set_title('Attention Weight Distribution by Component')
        ax.set_ylabel('Attention Weight')
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

# ============================================================================
# MAIN EXECUTION PIPELINE
# ============================================================================
def main():
    """Main execution pipeline for the orthogonal ensemble"""
    print("\n" + "="*80)
    print("🚀 6-COMPONENT ORTHOGONAL ENSEMBLE FOR PHOSPHORYLATION PREDICTION")
    print("="*80)
    
    # Set random seeds
    np.random.seed(Config.RANDOM_SEED)
    torch.manual_seed(Config.RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(Config.RANDOM_SEED)
    
    # Initialize progress tracker with the proper implementation from Section 0
    # This is the actual ProgressTracker class from your code
    class ProgressTracker:
        """Progress tracking with checkpoint management"""
        
        def __init__(self, exp_dir: str, auto_cleanup: bool = True):
            self.exp_dir = exp_dir
            self.auto_cleanup = auto_cleanup
            self.progress_file = os.path.join(exp_dir, 'progress_tracker.json')
            self.start_time = datetime.now()
            
            # Create directory structure
            self._create_directories()
            
            # Load or initialize progress
            self.progress = self._load_progress()
            
            # Memory monitoring
            self.memory_threshold = 0.8  # 80% memory usage triggers cleanup
            
        def _create_directories(self):
            """Create all required directories"""
            directories = [
                self.exp_dir,
                os.path.join(self.exp_dir, 'checkpoints'),
                os.path.join(self.exp_dir, 'orthogonal_ensemble'),
                os.path.join(self.exp_dir, 'orthogonal_ensemble/plots'),
                os.path.join(self.exp_dir, 'orthogonal_ensemble/models'),
                os.path.join(self.exp_dir, 'logs'),
            ]
            
            for directory in directories:
                os.makedirs(directory, exist_ok=True)
        
        def _load_progress(self) -> Dict:
            """Load progress from file if exists"""
            if os.path.exists(self.progress_file):
                with open(self.progress_file, 'r') as f:
                    return json.load(f)
            else:
                return {
                    'experiment_start': self.start_time.isoformat(),
                    'completed_steps': {},
                    'checkpoints': {},
                    'metadata': {}
                }
        
        def _save_progress(self):
            """Save progress to file"""
            with open(self.progress_file, 'w') as f:
                json.dump(self.progress, f, indent=2, default=str)
        
        def mark_completed(self, step_name: str, metadata: Dict = None, checkpoint_data: Any = None):
            """Mark a step as completed and optionally save checkpoint"""
            completion_time = datetime.now()
            self.progress['completed_steps'][step_name] = {
                'completed_at': completion_time.isoformat(),
                'duration_seconds': (completion_time - self.start_time).total_seconds(),
                'metadata': metadata or {}
            }
            
            if checkpoint_data is not None:
                checkpoint_path = os.path.join(
                    self.exp_dir, 'checkpoints', f'{step_name.replace(" ", "_").lower()}.pkl'
                )
                with open(checkpoint_path, 'wb') as f:
                    pickle.dump(checkpoint_data, f, protocol=4)
                self.progress['checkpoints'][step_name] = checkpoint_path
            
            self._save_progress()
        
        def is_completed(self, step_name: str) -> bool:
            """Check if a step is already completed"""
            return step_name in self.progress['completed_steps']
        
        def resume_from_checkpoint(self, step_name: str) -> Any:
            """Resume from a checkpoint if exists"""
            if step_name in self.progress['checkpoints']:
                checkpoint_path = self.progress['checkpoints'][step_name]
                if os.path.exists(checkpoint_path):
                    with open(checkpoint_path, 'rb') as f:
                        return pickle.load(f)
            return None
        
        def get_memory_usage(self) -> Dict:
            """Get current memory usage"""
            import psutil
            process = psutil.Process()
            memory_info = process.memory_info()
            
            return {
                'rss_mb': memory_info.rss / 1024 / 1024,  # Resident Set Size in MB
                'vms_mb': memory_info.vms / 1024 / 1024,  # Virtual Memory Size in MB
                'percent': process.memory_percent()
            }
    
    # Initialize progress tracker
    progress_tracker = ProgressTracker(Config.BASE_DIR, auto_cleanup=True)
    print(f"✓ Progress tracker initialized. Base directory: {Config.BASE_DIR}")
    
    # Initialize data loader
    data_loader = DataLoader(progress_tracker, Config)
    
    # Load all data
    print("\nStep 1: Loading data from checkpoints...")
    data = data_loader.load_all_data()
    
    # Extract sequence windows
    print("\nStep 2: Extracting sequence windows...")
    data_loader.extract_sequence_windows()
    
    # Prepare features
    print("\nStep 3: Preparing and standardizing features...")
    data_loader.prepare_features()
    
    # Tokenize sequences
    print("\nStep 4: Tokenizing sequences for transformer...")
    data_loader.tokenize_sequences()
    
    # Create train/validation split from training data
    print("\nStep 5: Creating train/validation split...")
    train_indices = data['train_indices']
    np.random.shuffle(train_indices)
    
    val_size = int(0.15 * len(train_indices))
    val_indices = train_indices[:val_size]
    train_indices_final = train_indices[val_size:]
    
    print(f"Train size: {len(train_indices_final)}")
    print(f"Validation size: {len(val_indices)}")
    print(f"Test size: {len(data['test_indices'])}")
    
    # Create datasets
    train_dataset = PhosphoDataset(train_indices_final, data_loader.data)
    val_dataset = PhosphoDataset(val_indices, data_loader.data)
    test_dataset = PhosphoDataset(data['test_indices'], data_loader.data)
    
    # Initialize model
    print("\nStep 6: Initializing orthogonal ensemble model...")
    model = OrthogonalEnsemble()
    print(f"Model initialized with {sum(p.numel() for p in model.parameters())} parameters")
    
    # Initialize trainer
    trainer = Trainer(model, data_loader, Config)
    
    # Phase 1: Individual component training
    print("\nStep 7: Phase 1 - Individual Component Training...")
    phase1_start = datetime.now()
    component_results = trainer.phase1_training(train_dataset, val_dataset)
    phase1_duration = (datetime.now() - phase1_start).total_seconds() / 60
    print(f"\nPhase 1 completed in {phase1_duration:.1f} minutes")
    
    # Phase 2: Fusion network training
    print("\nStep 8: Phase 2 - Fusion Network Training...")
    phase2_start = datetime.now()
    fusion_f1 = trainer.phase2_training(train_dataset, val_dataset)
    phase2_duration = (datetime.now() - phase2_start).total_seconds() / 60
    print(f"\nPhase 2 completed in {phase2_duration:.1f} minutes")
    
    # Phase 3: End-to-end fine-tuning
    print("\nStep 9: Phase 3 - End-to-End Fine-tuning...")
    phase3_start = datetime.now()
    final_f1 = trainer.phase3_training(train_dataset, val_dataset)
    phase3_duration = (datetime.now() - phase3_start).total_seconds() / 60
    print(f"\nPhase 3 completed in {phase3_duration:.1f} minutes")
    
    # Total training time
    total_duration = phase1_duration + phase2_duration + phase3_duration
    print(f"\nTotal training time: {total_duration:.1f} minutes ({total_duration/60:.1f} hours)")
    
    # Evaluate on test set
    print("\nStep 10: Final evaluation on test set...")
    test_loader = DataLoader(test_dataset, batch_size=Config.PHASE2_BATCH_SIZE, shuffle=False)
    evaluator = Evaluator(model, Config.DEVICE)
    test_results = evaluator.evaluate_on_test(test_loader)
    
    # Save results
    print("\nStep 11: Saving results...")
    results_dir = os.path.join(Config.BASE_DIR, 'orthogonal_ensemble_results')
    os.makedirs(results_dir, exist_ok=True)
    
    # Save model
    model_path = os.path.join(results_dir, 'best_model.pth')
    torch.save(model.state_dict(), model_path)
    print(f"✓ Model saved to {model_path}")
    
    # Save training results
    training_results = {
        'phase1_component_results': {k: v['f1'] for k, v in component_results.items()},
        'phase2_fusion_f1': fusion_f1,
        'phase3_final_f1': final_f1,
        'test_results': test_results['final_metrics'],
        'component_test_results': test_results['component_metrics'],
        'training_duration': {
            'phase1_minutes': phase1_duration,
            'phase2_minutes': phase2_duration,
            'phase3_minutes': phase3_duration,
            'total_minutes': total_duration
        },
        'config': {
            'phase1_epochs': Config.PHASE1_EPOCHS,
            'phase2_epochs': Config.PHASE2_EPOCHS,
            'phase3_epochs': Config.PHASE3_EPOCHS,
            'phase1_batch_size': Config.PHASE1_BATCH_SIZE,
            'phase2_batch_size': Config.PHASE2_BATCH_SIZE,
            'phase3_batch_size': Config.PHASE3_BATCH_SIZE
        }
    }
    
    # Add all trainer results
    training_results.update(trainer.results)
    
    results_path = os.path.join(results_dir, 'training_results.pkl')
    with open(results_path, 'wb') as f:
        pickle.dump(training_results, f)
    print(f"✓ Training results saved to {results_path}")
    
    # Save test predictions
    predictions_data = {
        'predictions': test_results['predictions'],
        'labels': test_results['labels'],
        'attention_weights': test_results['attention_weights'],
        'test_indices': data['test_indices']
    }
    
    predictions_path = os.path.join(results_dir, 'test_predictions.pkl')
    with open(predictions_path, 'wb') as f:
        pickle.dump(predictions_data, f)
    print(f"✓ Test predictions saved to {predictions_path}")
    
    # Generate visualizations
    print("\nStep 12: Generating visualizations...")
    
    # Training curves
    curves_path = os.path.join(results_dir, 'training_curves.png')
    evaluator.plot_training_curves(trainer.results, curves_path)
    print(f"✓ Training curves saved to {curves_path}")
    
    # Attention analysis
    attention_path = os.path.join(results_dir, 'attention_analysis.png')
    evaluator.plot_attention_analysis(test_results['attention_weights'], attention_path)
    print(f"✓ Attention analysis saved to {attention_path}")
    
    # Save checkpoint for future use
    checkpoint_data = {
        'model_state_dict': model.state_dict(),
        'scalers': data_loader.scalers,
        'pca_models': data_loader.pca_models,  # Save PCA models
        'training_results': training_results,
        'test_results': test_results,
        'config': Config.__dict__
    }
    
    progress_tracker.mark_completed(
        "orthogonal_ensemble",
        metadata={
            'final_test_f1': test_results['final_metrics']['f1'],
            'total_training_minutes': total_duration,
            'num_parameters': sum(p.numel() for p in model.parameters()),
            'pca_components': Config.PCA_COMPONENTS
        },
        checkpoint_data=checkpoint_data
    )
    
    # Print final summary
    print("\n" + "="*80)
    print("🎉 ORTHOGONAL ENSEMBLE TRAINING COMPLETE!")
    print("="*80)
    print(f"\nPERFORMANCE SUMMARY:")
    print(f"Phase 1 Best Component F1: {max(v['f1'] for v in component_results.values()):.4f}")
    print(f"Phase 2 Fusion F1: {fusion_f1:.4f}")
    print(f"Phase 3 Final Validation F1: {final_f1:.4f}")
    print(f"Test Set F1: {test_results['final_metrics']['f1']:.4f}")
    print(f"\nIMPROVEMENT:")
    print(f"Over best component: +{test_results['final_metrics']['f1'] - max(v['f1'] for v in component_results.values()):.4f}")
    print(f"Over current best (0.8025): +{test_results['final_metrics']['f1'] - 0.8025:.4f}")
    print(f"\nAll results saved to: {results_dir}")

# ============================================================================
# INFERENCE FUNCTIONS
# ============================================================================
def load_trained_model(model_path, scalers_path=None):
    """Load a trained orthogonal ensemble model"""
    # Initialize model
    model = OrthogonalEnsemble()
    
    # Load state dict
    state_dict = torch.load(model_path, map_location=Config.DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    
    # Load scalers if provided
    scalers = None
    if scalers_path:
        with open(scalers_path, 'rb') as f:
            scalers = pickle.load(f)
    
    return model, scalers

def predict_single_sample(model, scalers, sequence, position, tokenizer=None):
    """Predict phosphorylation for a single site"""
    # Extract features (you'll need to implement these based on your feature extraction code)
    # This is a placeholder - replace with actual feature extraction
    
    # For now, return a dummy prediction
    # In practice, you'd extract all features and run through the model
    return {"probability": 0.5, "prediction": 0, "attention_weights": [0.167]*6}

# ============================================================================
# EXECUTE MAIN
# ============================================================================
if __name__ == "__main__":
    main()


🚀 6-COMPONENT ORTHOGONAL ENSEMBLE FOR PHOSPHORYLATION PREDICTION
✓ Progress tracker initialized. Base directory: results/exp_3

Step 1: Loading data from checkpoints...

LOADING DATA FROM CHECKPOINTS

Loading from Section 1 checkpoint...
✓ Loaded 62120 samples
  Columns in df_final: ['Header', 'Sequence', 'SeqLength', 'UniProt ID', 'AA', 'Position', 'target']

Loading from Section 2 checkpoint...
✓ Loaded feature matrices:
  - AAC: (62120, 20)
  - DPC: (62120, 400)
  - TPC: (62120, 7996)
  - Physicochemical: (62120, 656)
  - Binary: (62120, 820)

Loading from Section 3 checkpoint...
✓ Loaded splits:
  - Train: 42845 samples
  - Test: 10122 samples

Step 2: Extracting sequence windows...

Extracting sequence windows...
✓ Extracted 62120 sequence windows

Step 3: Preparing and standardizing features...

Preparing features with Section 4 transformations...
  Processing AAC features (Polynomial interactions)...
    AAC: 20 → 210 features
  Processing DPC features (PCA-30)...
    DPC: 400 

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Tokenized sequences: torch.Size([62120, 64])

Step 5: Creating train/validation split...
Train size: 36419
Validation size: 6426
Test size: 10122

Step 6: Initializing orthogonal ensemble model...
Model initialized with 8215446 parameters

Step 7: Phase 1 - Individual Component Training...

PHASE 1: INDIVIDUAL COMPONENT TRAINING

Training AAC Component


Epoch 1/50: 100%|████████████████████████████████████████████████████████| 143/143 [00:04<00:00, 32.38it/s, loss=0.545]



Epoch 1/50
Train Loss: 0.5956 | Val Loss: 0.5676
Train F1: 0.6971 | Val F1: 0.7330
Val Metrics - Acc: 0.7087, Prec: 0.6796, Rec: 0.7954, AUC: 0.7692


...


Epoch 23/30
Train Loss: 0.0760 | Val Loss: 0.7486
Train F1: 0.9765 | Val F1: 0.7985
Val Metrics - Acc: 0.7947, Prec: 0.7882, Rec: 0.8090, AUC: 0.8621
Attention Weights - AAC: 0.071, DPC: 0.028, TPC: 0.169, Phys: 0.201, Binary: 0.244, Transformer: 0.288


Epoch 24/30: 100%|████████████████████████████████████████████████████████| 72/72 [00:18<00:00,  3.90it/s, loss=0.0487]



Epoch 24/30
Train Loss: 0.0753 | Val Loss: 0.7501
Train F1: 0.9773 | Val F1: 0.7983
Val Metrics - Acc: 0.7957, Prec: 0.7921, Rec: 0.8046, AUC: 0.8621
Attention Weights - AAC: 0.070, DPC: 0.028, TPC: 0.171, Phys: 0.204, Binary: 0.240, Transformer: 0.288

Early stopping triggered after 24 epochs

✓ Fusion network training complete. Best Val F1: 0.7993

Phase 2 completed in 8.8 minutes

Step 9: Phase 3 - End-to-End Fine-tuning...

PHASE 3: END-TO-END FINE-TUNING
✓ All component models unfrozen (except ESM-2 embeddings)


Epoch 1/100:   0%|                                                                             | 0/285 [00:00<?, ?it/s]


RuntimeError: Found dtype Float but expected Half

In [25]:
# ============================================================================
# PHASE 3 END-TO-END TRAINING PIPELINE
# ============================================================================
import os
import gc
import json
import pickle
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader as TorchDataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR
from torch.cuda.amp import autocast, GradScaler

# Transformers
from transformers import AutoTokenizer, AutoModel

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Progress tracking
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    """Improved configuration based on Phase 3 results analysis"""
    # Experiment settings
    EXPERIMENT_NAME = "improved_orthogonal_ensemble"
    RANDOM_SEED = 42
    BASE_DIR = "results/exp_3"
    
    # Model settings
    WINDOW_SIZE = 20
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Training settings - INCREASED EPOCHS
    TEST_EPOCHS = 1
    FULL_EPOCHS = 150  # Increased from 100
    
    # Batch sizes
    TEST_BATCH_SIZE = 64
    FULL_BATCH_SIZE = 128
    
    # IMPROVED LEARNING RATES - Key Changes
    PHASE3_LR = {
        'aac_net': 2e-5,        # Increased from 1e-5
        'dpc_net': 2e-5,        # Increased from 1e-5  
        'tpc_net': 3e-5,        # Increased from 5e-6 (6x increase!)
        'phys_net': 1e-5,       # Keep same (performing well)
        'binary_net': 2e-5,     # Increased from 1e-5
        'transformer': 5e-6,    # Increased from 2e-6
        'fusion': 2e-4          # Increased from 1e-4
    }
    
    # IMPROVED REGULARIZATION
    DROPOUT_RATES = {
        'aac': 0.15,      # Increased from 0.1
        'dpc': 0.25,      # Increased from 0.2
        'tpc': 0.2,       # Decreased from 0.3 (was too high)
        'phys': 0.2,      # Keep same
        'binary': 0.25,   # Increased from 0.2
        'transformer': 0.25,  # Decreased from 0.3
        'fusion': 0.15    # Increased from 0.1
    }
    
    # IMPROVED EARLY STOPPING
    EARLY_STOPPING_PATIENCE = 25  # Increased from 15
    
    # Transformer settings
    TRANSFORMER_MODEL = "facebook/esm2_t6_8M_UR50D"
    MAX_LENGTH = 64
    
# ============================================================================
# FIXED MULTI-OBJECTIVE LOSS
# ============================================================================
class MultiObjectiveLoss(nn.Module):
    """Improved loss with better component balancing"""
    def __init__(self, diversity_weight=0.05, component_weight=0.15):  # Changed weights
        super(ImprovedMultiObjectiveLoss, self).__init__()
        self.bce_loss = nn.BCELoss()
        self.diversity_weight = diversity_weight  # Reduced from 0.1
        self.component_weight = component_weight  # Increased from 0.05
        
    def forward(self, outputs, labels):
        """Calculate improved multi-objective loss"""
        # Ensure all tensors are Float32
        final_pred = outputs['final_prediction'].squeeze().float()
        labels = labels.float()
        
        # Main prediction loss
        main_loss = self.bce_loss(final_pred, labels)
        
        # IMPROVED: Weighted diversity loss (penalize extreme attention)
        attention_weights = outputs['attention_weights'].float()
        
        # Calculate attention entropy (higher entropy = more diverse)
        attention_entropy = -torch.sum(attention_weights * torch.log(attention_weights + 1e-8), dim=1)
        max_entropy = torch.log(torch.tensor(6.0))  # log(6) for 6 components
        diversity_loss = torch.mean(max_entropy - attention_entropy)  # Minimize deviation from max entropy
        
        # IMPROVED: Component-specific loss weighting
        component_losses = []
        component_weights = [1.5, 2.0, 3.0, 1.0, 2.0, 1.0]  # Higher weight for underperforming components
        
        components = ['aac', 'dpc', 'tpc', 'phys', 'binary', 'transformer']
        for i, (comp_name, pred) in enumerate(outputs['individual_predictions'].items()):
            pred_float = pred.squeeze().float()
            comp_loss = self.bce_loss(pred_float, labels)
            component_losses.append(component_weights[i] * comp_loss)
        
        avg_component_loss = torch.mean(torch.stack(component_losses))
        
        # Combined loss
        total_loss = (main_loss + 
                     self.diversity_weight * diversity_loss + 
                     self.component_weight * avg_component_loss)
        
        return {
            'total_loss': total_loss,
            'main_loss': main_loss,
            'diversity_loss': diversity_loss,
            'component_loss': avg_component_loss
        }

# ============================================================================
# DATA LOADER
# ============================================================================
class Phase3DataLoader:
    """Load data for Phase 3 training"""
    def __init__(self, base_dir=Config.BASE_DIR):
        self.base_dir = base_dir
        self.data = {}
        
    def load_checkpoint_data(self):
        """Load data from available checkpoints"""
        print("Loading data from checkpoints...")
        
        # Check for available checkpoints
        checkpoints_dir = os.path.join(self.base_dir, 'checkpoints')
        if not os.path.exists(checkpoints_dir):
            raise FileNotFoundError(f"Checkpoints directory not found: {checkpoints_dir}")
        
        # List available checkpoints
        available_checkpoints = [f for f in os.listdir(checkpoints_dir) if f.endswith('.pkl')]
        print(f"Available checkpoints: {available_checkpoints}")
        
        # Load essential checkpoints based on what's available
        checkpoint_data = {}
        
        # 1. Load data_loading checkpoint
        data_checkpoint_path = os.path.join(checkpoints_dir, 'data_loading.pkl')
        if os.path.exists(data_checkpoint_path):
            with open(data_checkpoint_path, 'rb') as f:
                checkpoint_data['data_loading'] = pickle.load(f)
            print("✓ Loaded data_loading checkpoint")
        else:
            raise FileNotFoundError("data_loading.pkl checkpoint not found")
        
        # 2. Load feature_extraction checkpoint  
        feature_checkpoint_path = os.path.join(checkpoints_dir, 'feature_extraction.pkl')
        if os.path.exists(feature_checkpoint_path):
            with open(feature_checkpoint_path, 'rb') as f:
                checkpoint_data['feature_extraction'] = pickle.load(f)
            print("✓ Loaded feature_extraction checkpoint")
        else:
            raise FileNotFoundError("feature_extraction.pkl checkpoint not found")
        
        # 3. Load data_splitting checkpoint
        split_checkpoint_path = os.path.join(checkpoints_dir, 'data_splitting.pkl')
        if os.path.exists(split_checkpoint_path):
            with open(split_checkpoint_path, 'rb') as f:
                checkpoint_data['data_splitting'] = pickle.load(f)
            print("✓ Loaded data_splitting checkpoint")
        else:
            raise FileNotFoundError("data_splitting.pkl checkpoint not found")
        
        # 4. Look for any existing model checkpoints (optional)
        model_checkpoints = [f for f in available_checkpoints if 'ensemble' in f or 'phase' in f]
        if model_checkpoints:
            print(f"Found existing model checkpoints: {model_checkpoints}")
            # Load the most recent one
            latest_model = sorted(model_checkpoints)[-1]
            model_checkpoint_path = os.path.join(checkpoints_dir, latest_model)
            with open(model_checkpoint_path, 'rb') as f:
                checkpoint_data['model_checkpoint'] = pickle.load(f)
            print(f"✓ Loaded model checkpoint: {latest_model}")
        else:
            print("⚠ No existing model checkpoints found, will start from scratch")
            checkpoint_data['model_checkpoint'] = None
        
        return checkpoint_data
    
    def prepare_datasets(self, checkpoint_data):
        """Prepare datasets from checkpoint data"""
        print("Preparing datasets from checkpoints...")
        
        # Extract data from checkpoints
        data_checkpoint = checkpoint_data['data_loading']
        feature_checkpoint = checkpoint_data['feature_extraction']
        split_checkpoint = checkpoint_data['data_splitting']
        
        # Get the main dataframe
        df_final = data_checkpoint['df_final']
        print(f"✓ Loaded df_final with {len(df_final)} samples")
        
        # Get feature matrices
        feature_matrices = feature_checkpoint['feature_matrices']
        
        # Prepare transformed features for Section 4 methods
        self.prepare_transformed_features(feature_matrices, split_checkpoint)
        
        # Get train/test splits
        if 'splits' in split_checkpoint:
            splits = split_checkpoint['splits']
            train_indices = splits['train_indices']
            test_indices = splits['test_indices']
        else:
            # Fallback for older checkpoint format
            train_indices = split_checkpoint.get('train_indices')
            test_indices = split_checkpoint.get('test_indices')
        
        if train_indices is None or test_indices is None:
            raise ValueError("Could not find train/test indices in split checkpoint")
        
        # Create train/val split from training data
        np.random.seed(Config.RANDOM_SEED)
        np.random.shuffle(train_indices)
        val_size = int(0.15 * len(train_indices))
        val_indices = train_indices[:val_size]
        train_indices_final = train_indices[val_size:]
        
        print(f"✓ Split preparation:")
        print(f"  Train: {len(train_indices_final)} samples")
        print(f"  Val: {len(val_indices)} samples")
        print(f"  Test: {len(test_indices)} samples")
        
        # Prepare sequence data
        self.prepare_sequence_data(df_final)
        
        # Create datasets
        train_dataset = PhosphoDataset(train_indices_final, self.data)
        val_dataset = PhosphoDataset(val_indices, self.data)
        test_dataset = PhosphoDataset(test_indices, self.data)
        
        return {
            'train_dataset': train_dataset,
            'val_dataset': val_dataset,
            'test_dataset': test_dataset,
            'model_checkpoint': checkpoint_data.get('model_checkpoint'),
            'data_info': {
                'total_samples': len(df_final),
                'train_size': len(train_indices_final),
                'val_size': len(val_indices),
                'test_size': len(test_indices)
            }
        }
    
    def prepare_transformed_features(self, feature_matrices, split_checkpoint):
        """Apply Section 4 transformations to features"""
        print("Applying feature transformations...")
        
        # Get training indices for fitting transformers
        if 'splits' in split_checkpoint:
            train_indices = split_checkpoint['splits']['train_indices']
        else:
            train_indices = split_checkpoint.get('train_indices')
        
        # Import transformation libraries
        from sklearn.decomposition import PCA
        from sklearn.preprocessing import PolynomialFeatures, StandardScaler
        from sklearn.feature_selection import SelectKBest, mutual_info_classif, VarianceThreshold
        
        # Get target from the main DataFrame (not feature matrix)
        # Based on Section 4, target should be extracted from df_final
        data_checkpoint_path = os.path.join(self.base_dir, 'checkpoints', 'data_loading.pkl')
        with open(data_checkpoint_path, 'rb') as f:
            data_checkpoint = pickle.load(f)
        
        df_final = data_checkpoint['df_final']
        target = df_final['target'].values
        print(f"✓ Extracted target from df_final: {len(target)} samples, positive ratio: {np.mean(target):.3f}")
        
        # Convert DataFrames to numpy arrays
        aac_features = feature_matrices['aac'].select_dtypes(include=[np.number]).values
        dpc_features = feature_matrices['dpc'].select_dtypes(include=[np.number]).values
        tpc_features = feature_matrices['tpc'].select_dtypes(include=[np.number]).values
        phys_features = feature_matrices['physicochemical'].select_dtypes(include=[np.number]).values
        binary_features = feature_matrices['binary'].select_dtypes(include=[np.number]).values
        
        # ========== AAC: Polynomial Features ==========
        print("  Processing AAC features (Polynomial interactions)...")
        poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
        poly.fit(aac_features[train_indices])
        self.data['aac_scaled'] = poly.transform(aac_features)
        print(f"    AAC: {aac_features.shape[1]} → {self.data['aac_scaled'].shape[1]} features")
        
        # ========== DPC: PCA with 30 components ==========
        print("  Processing DPC features (PCA-30)...")
        dpc_scaler = StandardScaler()
        dpc_scaler.fit(dpc_features[train_indices])
        dpc_scaled = dpc_scaler.transform(dpc_features)
        
        dpc_pca = PCA(n_components=30, random_state=Config.RANDOM_SEED)
        dpc_pca.fit(dpc_scaled[train_indices])
        self.data['dpc_scaled'] = dpc_pca.transform(dpc_scaled)
        print(f"    DPC: {dpc_features.shape[1]} → {self.data['dpc_scaled'].shape[1]} features")
        
        # ========== TPC: PCA with 50 components ==========
        print("  Processing TPC features (PCA-50)...")
        tpc_scaler = StandardScaler()
        tpc_scaler.fit(tpc_features[train_indices])
        tpc_scaled = tpc_scaler.transform(tpc_features)
        
        tpc_pca = PCA(n_components=50, random_state=Config.RANDOM_SEED)
        tpc_pca.fit(tpc_scaled[train_indices])
        self.data['tpc_scaled'] = tpc_pca.transform(tpc_scaled)
        print(f"    TPC: {tpc_features.shape[1]} → {self.data['tpc_scaled'].shape[1]} features")
        
        # ========== Physicochemical: Mutual Information Selection (500 features) ==========
        print("  Processing Physicochemical features (Mutual Info 500)...")
        n_features = min(500, phys_features.shape[1])  # Don't exceed available features
        phys_selector = SelectKBest(score_func=mutual_info_classif, k=n_features)
        phys_selector.fit(phys_features[train_indices], target[train_indices])
        self.data['physicochemical_scaled'] = phys_selector.transform(phys_features)
        print(f"    Physicochemical: {phys_features.shape[1]} → {self.data['physicochemical_scaled'].shape[1]} features")
        
        # ========== Binary: Variance Threshold + PCA (200 components) ==========
        print("  Processing Binary features (Variance threshold + PCA-200)...")
        # Step 1: Variance threshold
        var_selector = VarianceThreshold(threshold=0.01)
        var_selector.fit(binary_features[train_indices])
        binary_var_selected = var_selector.transform(binary_features)
        
        # Step 2: Standardize
        binary_scaler = StandardScaler()
        binary_scaler.fit(binary_var_selected[train_indices])
        binary_scaled = binary_scaler.transform(binary_var_selected)
        
        # Step 3: PCA
        n_components = min(200, binary_scaled.shape[1])  # Don't exceed available features
        binary_pca = PCA(n_components=n_components, random_state=Config.RANDOM_SEED)
        binary_pca.fit(binary_scaled[train_indices])
        self.data['binary_scaled'] = binary_pca.transform(binary_scaled)
        print(f"    Binary: {binary_features.shape[1]} → {binary_var_selected.shape[1]} (variance) → {self.data['binary_scaled'].shape[1]} features")
        
        # Store target
        self.data['target'] = target
        
        print("✓ Feature transformations completed")
    
    def prepare_sequence_data(self, df_final):
        """Extract and tokenize sequence windows"""
        print("Preparing sequence data...")
        
        # Extract sequence windows around phosphorylation sites
        sequences = df_final['Sequence'].values
        positions = df_final['Position'].values
        
        window_sequences = []
        for seq, pos in zip(sequences, positions):
            # Get window around phosphorylation site
            start = max(0, pos - Config.WINDOW_SIZE // 2)
            end = min(len(seq), pos + Config.WINDOW_SIZE // 2 + 1)
            
            window = seq[start:end]
            
            # Pad if necessary
            if len(window) < Config.WINDOW_SIZE + 1:
                if start == 0:
                    window = 'X' * (Config.WINDOW_SIZE + 1 - len(window)) + window
                else:
                    window = window + 'X' * (Config.WINDOW_SIZE + 1 - len(window))
            
            window_sequences.append(window)
        
        # Tokenize sequences
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(Config.TRANSFORMER_MODEL)
        
        tokenized = tokenizer(
            window_sequences,
            padding='max_length',
            truncation=True,
            max_length=Config.MAX_LENGTH,
            return_tensors='pt'
        )
        
        self.data['input_ids'] = tokenized['input_ids']
        self.data['attention_mask'] = tokenized['attention_mask']
        
        print(f"✓ Tokenized {len(window_sequences)} sequence windows")

# ============================================================================
# MODELS (Copy from your original code)
# ============================================================================
class AACNetwork(nn.Module):
    """Neural network for Amino Acid Composition features"""
    def __init__(self, input_dim=210, dropout_rate=0.1):
        super(AACNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

class DPCNetwork(nn.Module):
    """Neural network for Dipeptide Composition features"""
    def __init__(self, input_dim=30, dropout_rate=0.2):
        super(DPCNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

class TPCNetwork(nn.Module):
    """Neural network for Tripeptide Composition features"""
    def __init__(self, input_dim=50, dropout_rate=0.3):
        super(TPCNetwork, self).__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

class PhysicochemicalNetwork(nn.Module):
    """Neural network for Physicochemical features"""
    def __init__(self, input_dim=500, dropout_rate=0.2):
        super(PhysicochemicalNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

class BinaryNetwork(nn.Module):
    """Neural network for Binary encoding features"""
    def __init__(self, input_dim=200, dropout_rate=0.2):
        super(BinaryNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

class SequenceTransformer(nn.Module):
    """ESM-2 based transformer for sequence understanding"""
    def __init__(self, dropout_rate=0.3):
        super(SequenceTransformer, self).__init__()
        
        # Load pre-trained ESM-2 model
        self.esm_model = AutoModel.from_pretrained(Config.TRANSFORMER_MODEL)
        
        # Freeze ESM-2 embeddings
        for param in self.esm_model.embeddings.parameters():
            param.requires_grad = False
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(320, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.esm_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        
        hidden_states = outputs.last_hidden_state
        batch_size = hidden_states.size(0)
        center_idx = hidden_states.size(1) // 2
        center_repr = hidden_states[:, center_idx, :]
        
        return self.classifier(center_repr)

class FusionNetwork(nn.Module):
    """Improved fusion network with temperature scaling"""
    def __init__(self, dropout_rate=0.15, temperature=2.0):  # Add temperature parameter
        super(FusionNetwork, self).__init__()
        self.temperature = temperature
        
        # Larger encoding dimension for better representation
        self.prediction_encoder = nn.Sequential(
            nn.Linear(6, 64),  # Increased from 32
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 32),
            nn.LayerNorm(32), 
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        
        # Temperature-scaled attention
        self.attention = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(dropout_rate/2),
            nn.Linear(16, 6)
            # Note: Softmax will be applied with temperature
        )
        
        # Improved predictor
        self.predictor = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )
    
    def forward(self, component_predictions):
        # Encode component predictions
        encoded = self.prediction_encoder(component_predictions)
        
        # Calculate temperature-scaled attention weights
        attention_logits = self.attention(encoded)
        attention_weights = F.softmax(attention_logits / self.temperature, dim=1)
        
        # Apply attention to original predictions
        weighted_preds = attention_weights * component_predictions
        
        # Re-encode weighted predictions
        weighted_encoded = self.prediction_encoder(weighted_preds)
        
        # Final prediction
        final_pred = self.predictor(weighted_encoded)
        
        return final_pred, attention_weights

class OrthogonalEnsemble(nn.Module):
    """Complete 6-component orthogonal ensemble"""
    def __init__(self):
        super(OrthogonalEnsemble, self).__init__()
        
        self.aac_net = AACNetwork(input_dim=210, dropout_rate=0.1)
        self.dpc_net = DPCNetwork(input_dim=30, dropout_rate=0.2)
        self.tpc_net = TPCNetwork(input_dim=50, dropout_rate=0.3)
        self.phys_net = PhysicochemicalNetwork(input_dim=500, dropout_rate=0.2)
        self.binary_net = BinaryNetwork(input_dim=200, dropout_rate=0.2)
        self.transformer = SequenceTransformer(0.3)
        self.fusion_net = FusionNetwork(0.1)
        
        self.component_names = ['aac', 'dpc', 'tpc', 'phys', 'binary', 'transformer']
        
    def forward(self, aac_features, dpc_features, tpc_features, phys_features, 
                binary_features, input_ids, attention_mask):
        
        aac_pred = self.aac_net(aac_features)
        dpc_pred = self.dpc_net(dpc_features)
        tpc_pred = self.tpc_net(tpc_features)
        phys_pred = self.phys_net(phys_features)
        binary_pred = self.binary_net(binary_features)
        transformer_pred = self.transformer(input_ids, attention_mask)
        
        component_predictions = torch.cat([
            aac_pred, dpc_pred, tpc_pred, phys_pred, binary_pred, transformer_pred
        ], dim=1)
        
        final_pred, attention_weights = self.fusion_net(component_predictions)
        
        return {
            'final_prediction': final_pred,
            'component_predictions': component_predictions,
            'attention_weights': attention_weights,
            'individual_predictions': {
                'aac': aac_pred,
                'dpc': dpc_pred,
                'tpc': tpc_pred,
                'phys': phys_pred,
                'binary': binary_pred,
                'transformer': transformer_pred
            }
        }
    
    def unfreeze_components(self):
        """Unfreeze all component models except ESM-2 embeddings"""
        for name in self.component_names:
            if name == 'transformer':
                for param in self.transformer.parameters():
                    param.requires_grad = True
                for param in self.transformer.esm_model.embeddings.parameters():
                    param.requires_grad = False
            else:
                component = getattr(self, f"{name}_net")
                for param in component.parameters():
                    param.requires_grad = True

# ============================================================================
# PHASE 3 TRAINER
# ============================================================================
class Phase3Trainer:
    """Phase 3 End-to-End Trainer with fixed dtype issues"""
    def __init__(self, model, device=Config.DEVICE):
        self.model = model.to(device)
        self.device = device
        self.results = defaultdict(list)
        
    def calculate_metrics(self, y_true, y_pred, y_prob):
        """Calculate evaluation metrics"""
        return {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'auc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.0
        }
    
    def run_pipeline_test(self, train_loader, val_loader):
        """Run a quick 1-epoch test to check for errors"""
        print("\n" + "="*60)
        print("RUNNING PIPELINE TEST (1 EPOCH)")
        print("="*60)
        
        # Unfreeze components
        self.model.unfreeze_components()
        
        # Setup optimizers
        param_groups = [
            {'params': self.model.aac_net.parameters(), 'lr': Config.PHASE3_LR['aac_net']},
            {'params': self.model.dpc_net.parameters(), 'lr': Config.PHASE3_LR['dpc_net']},
            {'params': self.model.tpc_net.parameters(), 'lr': Config.PHASE3_LR['tpc_net']},
            {'params': self.model.phys_net.parameters(), 'lr': Config.PHASE3_LR['phys_net']},
            {'params': self.model.binary_net.parameters(), 'lr': Config.PHASE3_LR['binary_net']},
            {'params': self.model.transformer.parameters(), 'lr': Config.PHASE3_LR['transformer']},
            {'params': self.model.fusion_net.parameters(), 'lr': Config.PHASE3_LR['fusion']}
        ]
        
        optimizer = AdamW(param_groups, weight_decay=1e-4)
        criterion = MultiObjectiveLoss(diversity_weight=0.1, component_weight=0.05)
        
        # Test training loop
        self.model.train()
        train_loss = 0
        processed_batches = 0
        
        try:
            print("Testing training loop...")
            progress_bar = tqdm(train_loader, desc="Pipeline Test - Train")
            
            for i, batch in enumerate(progress_bar):
                # Test with regular precision (no autocast for test)
                outputs = self.model(
                    batch['aac'].to(self.device),
                    batch['dpc'].to(self.device),
                    batch['tpc'].to(self.device),
                    batch['phys'].to(self.device),
                    batch['binary'].to(self.device),
                    batch['input_ids'].to(self.device),
                    batch['attention_mask'].to(self.device)
                )
                
                labels = batch['label'].to(self.device)
                loss_dict = criterion(outputs, labels)
                loss = loss_dict['total_loss']
                
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                
                train_loss += loss.item()
                processed_batches += 1
                
                progress_bar.set_postfix({
                    'loss': loss.item(),
                    'main': loss_dict['main_loss'].item(),
                    'batch': f"{i+1}/{len(train_loader)}"
                })
                
                # Test only first few batches
                if i >= 2:
                    break
            
            print(f"✓ Training loop test passed! Processed {processed_batches} batches")
            
            # Test validation loop
            self.model.eval()
            val_loss = 0
            processed_val_batches = 0
            
            print("Testing validation loop...")
            with torch.no_grad():
                progress_bar = tqdm(val_loader, desc="Pipeline Test - Val")
                for i, batch in enumerate(progress_bar):
                    outputs = self.model(
                        batch['aac'].to(self.device),
                        batch['dpc'].to(self.device),
                        batch['tpc'].to(self.device),
                        batch['phys'].to(self.device),
                        batch['binary'].to(self.device),
                        batch['input_ids'].to(self.device),
                        batch['attention_mask'].to(self.device)
                    )
                    
                    labels = batch['label'].to(self.device)
                    loss_dict = criterion(outputs, labels)
                    
                    val_loss += loss_dict['total_loss'].item()
                    processed_val_batches += 1
                    
                    progress_bar.set_postfix({
                        'loss': loss_dict['total_loss'].item(),
                        'batch': f"{i+1}/{len(val_loader)}"
                    })
                    
                    # Test only first few batches
                    if i >= 2:
                        break
            
            print(f"✓ Validation loop test passed! Processed {processed_val_batches} batches")
            print(f"✓ Pipeline test completed successfully!")
            print(f"Average train loss: {train_loss/processed_batches:.4f}")
            print(f"Average val loss: {val_loss/processed_val_batches:.4f}")
            
            return True
            
        except Exception as e:
            print(f"❌ Pipeline test failed with error: {str(e)}")
            import traceback
            traceback.print_exc()
            return False
    
    def train_full(self, train_loader, val_loader, epochs=Config.FULL_EPOCHS):
        """Run full Phase 3 training"""
        print("\n" + "="*60)
        print("STARTING FULL PHASE 3 TRAINING")
        print("="*60)
        
        # Unfreeze components
        self.model.unfreeze_components()
        print("✓ All component models unfrozen (except ESM-2 embeddings)")
        
        # Setup optimizers
        param_groups = [
            {'params': self.model.aac_net.parameters(), 'lr': Config.PHASE3_LR['aac_net']},
            {'params': self.model.dpc_net.parameters(), 'lr': Config.PHASE3_LR['dpc_net']},
            {'params': self.model.tpc_net.parameters(), 'lr': Config.PHASE3_LR['tpc_net']},
            {'params': self.model.phys_net.parameters(), 'lr': Config.PHASE3_LR['phys_net']},
            {'params': self.model.binary_net.parameters(), 'lr': Config.PHASE3_LR['binary_net']},
            {'params': self.model.transformer.parameters(), 'lr': Config.PHASE3_LR['transformer']},
            {'params': self.model.fusion_net.parameters(), 'lr': Config.PHASE3_LR['fusion']}
        ]
        
        optimizer = AdamW(param_groups, weight_decay=1e-4)
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
        criterion = MultiObjectiveLoss(diversity_weight=0.1, component_weight=0.05)
        
        # Training metrics
        best_val_f1 = 0
        patience = 0
        train_losses = []
        val_losses = []
        val_f1_scores = []
        component_f1_history = defaultdict(list)
        
        for epoch in range(epochs):
            # Training phase
            self.model.train()
            train_loss = 0
            train_main_loss = 0
            train_diversity_loss = 0
            train_component_loss = 0
            all_preds = []
            all_labels = []
            
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
            
            for batch in progress_bar:
                # Regular precision training (no autocast to avoid dtype issues)
                outputs = self.model(
                    batch['aac'].to(self.device),
                    batch['dpc'].to(self.device),
                    batch['tpc'].to(self.device),
                    batch['phys'].to(self.device),
                    batch['binary'].to(self.device),
                    batch['input_ids'].to(self.device),
                    batch['attention_mask'].to(self.device)
                )
                
                labels = batch['label'].to(self.device)
                loss_dict = criterion(outputs, labels)
                loss = loss_dict['total_loss']
                
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                
                train_loss += loss.item()
                train_main_loss += loss_dict['main_loss'].item()
                train_diversity_loss += loss_dict['diversity_loss'].item()
                train_component_loss += loss_dict['component_loss'].item()
                
                all_preds.extend(outputs['final_prediction'].detach().cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
                progress_bar.set_postfix({
                    'loss': loss.item(),
                    'main': loss_dict['main_loss'].item(),
                    'div': loss_dict['diversity_loss'].item()
                })
            
            # Calculate training metrics
            train_loss /= len(train_loader)
            train_main_loss /= len(train_loader)
            train_diversity_loss /= len(train_loader)
            train_component_loss /= len(train_loader)
            
            train_preds_binary = (np.array(all_preds) > 0.5).astype(int).flatten()
            train_metrics = self.calculate_metrics(all_labels, train_preds_binary, np.array(all_preds).flatten())
            
            # Validation phase
            self.model.eval()
            val_loss = 0
            val_preds = []
            val_labels = []
            val_component_preds = defaultdict(list)
            
            with torch.no_grad():
                for batch in val_loader:
                    outputs = self.model(
                        batch['aac'].to(self.device),
                        batch['dpc'].to(self.device),
                        batch['tpc'].to(self.device),
                        batch['phys'].to(self.device),
                        batch['binary'].to(self.device),
                        batch['input_ids'].to(self.device),
                        batch['attention_mask'].to(self.device)
                    )
                    
                    labels = batch['label'].to(self.device)
                    loss_dict = criterion(outputs, labels)
                    
                    val_loss += loss_dict['total_loss'].item()
                    val_preds.extend(outputs['final_prediction'].cpu().numpy())
                    val_labels.extend(labels.cpu().numpy())
                    
                    # Store component predictions
                    for comp_name, comp_pred in outputs['individual_predictions'].items():
                        val_component_preds[comp_name].extend(comp_pred.cpu().numpy())
            
            # Calculate validation metrics
            val_loss /= len(val_loader)
            val_preds_binary = (np.array(val_preds) > 0.5).astype(int).flatten()
            val_metrics = self.calculate_metrics(val_labels, val_preds_binary, np.array(val_preds).flatten())
            
            # Calculate component metrics
            for comp_name, comp_preds in val_component_preds.items():
                comp_preds_binary = (np.array(comp_preds) > 0.5).astype(int).flatten()
                comp_metrics = self.calculate_metrics(val_labels, comp_preds_binary, np.array(comp_preds).flatten())
                component_f1_history[comp_name].append(comp_metrics['f1'])
            
            # Update scheduler
            scheduler.step()
            
            # Store metrics
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            val_f1_scores.append(val_metrics['f1'])
            
            # Print epoch summary
            print(f"\nEpoch {epoch+1}/{epochs}")
            print(f"Train Loss: {train_loss:.4f} (Main: {train_main_loss:.4f}, Div: {train_diversity_loss:.4f}, Comp: {train_component_loss:.4f})")
            print(f"Val Loss: {val_loss:.4f}")
            print(f"Train F1: {train_metrics['f1']:.4f} | Val F1: {val_metrics['f1']:.4f}")
            print(f"Val Metrics - Acc: {val_metrics['accuracy']:.4f}, Prec: {val_metrics['precision']:.4f}, Rec: {val_metrics['recall']:.4f}, AUC: {val_metrics['auc']:.4f}")
            
            # Print component F1 scores
            print("Component F1 Scores:")
            for comp_name in ['aac', 'dpc', 'tpc', 'phys', 'binary', 'transformer']:
                if comp_name in component_f1_history and len(component_f1_history[comp_name]) > 0:
                    print(f"  {comp_name}: {component_f1_history[comp_name][-1]:.4f}")
            
            # Early stopping
            if val_metrics['f1'] > best_val_f1:
                best_val_f1 = val_metrics['f1']
                patience = 0
                # Save best model
                best_model_state = self.model.state_dict()
                best_epoch = epoch + 1
            else:
                patience += 1
                if patience >= Config.EARLY_STOPPING_PATIENCE:
                    print(f"\nEarly stopping triggered after {epoch+1} epochs")
                    break
        
        # Load best model
        self.model.load_state_dict(best_model_state)
        
        # Store results
        self.results['e2e_train_losses'] = train_losses
        self.results['e2e_val_losses'] = val_losses
        self.results['e2e_val_f1_scores'] = val_f1_scores
        self.results['e2e_best_f1'] = best_val_f1
        self.results['e2e_best_epoch'] = best_epoch
        self.results['component_f1_history'] = dict(component_f1_history)
        
        print(f"\n✓ End-to-end training complete. Best Val F1: {best_val_f1:.4f} (Epoch {best_epoch})")
        
        return best_val_f1

# ============================================================================
# DATASET CLASS
# ============================================================================
class PhosphoDataset(Dataset):
    """Dataset for phosphorylation prediction"""
    def __init__(self, indices, data_dict):
        self.indices = indices
        self.data = data_dict
        
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        actual_idx = self.indices[idx]
        
        sample = {
            'aac': torch.tensor(self.data['aac_scaled'][actual_idx], dtype=torch.float32),
            'dpc': torch.tensor(self.data['dpc_scaled'][actual_idx], dtype=torch.float32),
            'tpc': torch.tensor(self.data['tpc_scaled'][actual_idx], dtype=torch.float32),
            'phys': torch.tensor(self.data['physicochemical_scaled'][actual_idx], dtype=torch.float32),
            'binary': torch.tensor(self.data['binary_scaled'][actual_idx], dtype=torch.float32),
            'input_ids': self.data['input_ids'][actual_idx],
            'attention_mask': self.data['attention_mask'][actual_idx],
            'label': torch.tensor(self.data['target'][actual_idx], dtype=torch.float32)
        }
        
        return sample

# ============================================================================
# EVALUATION
# ============================================================================
class Evaluator:
    """Evaluation tools"""
    def __init__(self, model, device):
        self.model = model
        self.device = device
    
    def evaluate_on_test(self, test_loader):
        """Evaluate model on test set"""
        print("\n" + "="*60)
        print("FINAL EVALUATION ON TEST SET")
        print("="*60)
        
        self.model.eval()
        all_preds = []
        all_labels = []
        all_attention_weights = []
        component_preds = defaultdict(list)
        
        with torch.no_grad():
            for batch in tqdm(test_loader, desc="Evaluating"):
                outputs = self.model(
                    batch['aac'].to(self.device),
                    batch['dpc'].to(self.device),
                    batch['tpc'].to(self.device),
                    batch['phys'].to(self.device),
                    batch['binary'].to(self.device),
                    batch['input_ids'].to(self.device),
                    batch['attention_mask'].to(self.device)
                )
                
                all_preds.extend(outputs['final_prediction'].cpu().numpy())
                all_labels.extend(batch['label'].numpy())
                all_attention_weights.extend(outputs['attention_weights'].cpu().numpy())
                
                for comp_name, comp_pred in outputs['individual_predictions'].items():
                    component_preds[comp_name].extend(comp_pred.cpu().numpy())
        
        # Convert to arrays
        all_preds = np.array(all_preds).flatten()
        all_labels = np.array(all_labels)
        all_attention_weights = np.array(all_attention_weights)
        
        # Calculate metrics
        preds_binary = (all_preds > 0.5).astype(int)
        final_metrics = self.calculate_metrics(all_labels, preds_binary, all_preds)
        
        print("\nFINAL TEST RESULTS:")
        print(f"Accuracy: {final_metrics['accuracy']:.4f}")
        print(f"Precision: {final_metrics['precision']:.4f}")
        print(f"Recall: {final_metrics['recall']:.4f}")
        print(f"F1 Score: {final_metrics['f1']:.4f}")
        print(f"AUC-ROC: {final_metrics['auc']:.4f}")
        
        # Component performance
        print("\nCOMPONENT PERFORMANCE:")
        component_metrics = {}
        for comp_name, comp_preds_list in component_preds.items():
            comp_preds_array = np.array(comp_preds_list).flatten()
            comp_preds_binary = (comp_preds_array > 0.5).astype(int)
            comp_metrics = self.calculate_metrics(all_labels, comp_preds_binary, comp_preds_array)
            component_metrics[comp_name] = comp_metrics
            print(f"{comp_name}: F1 = {comp_metrics['f1']:.4f}, AUC = {comp_metrics['auc']:.4f}")
        
        # Average attention weights
        avg_attention = np.mean(all_attention_weights, axis=0)
        print("\nAVERAGE ATTENTION WEIGHTS:")
        comp_names = ['AAC', 'DPC', 'TPC', 'Phys', 'Binary', 'Transformer']
        for i, name in enumerate(comp_names):
            print(f"{name}: {avg_attention[i]:.3f}")
        
        return {
            'final_metrics': final_metrics,
            'component_metrics': component_metrics,
            'predictions': all_preds,
            'labels': all_labels,
            'attention_weights': all_attention_weights
        }
    
    def calculate_metrics(self, y_true, y_pred, y_prob):
        """Calculate evaluation metrics"""
        return {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'auc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.0
        }

# ============================================================================
# MAIN PHASE 3 PIPELINE
# ============================================================================
def run_phase3_pipeline(test_only=True, run_full=False):
    """
    Main Phase 3 pipeline
    
    Args:
        test_only: If True, only run pipeline test (1 epoch)
        run_full: If True, run full training after successful test
    """
    print("=" * 80)
    print("🚀 PHASE 3 END-TO-END TRAINING PIPELINE")
    print("=" * 80)
    
    # Set random seeds
    np.random.seed(Config.RANDOM_SEED)
    torch.manual_seed(Config.RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(Config.RANDOM_SEED)
    
    print(f"Device: {Config.DEVICE}")
    
    # Step 1: Load data
    print("\nStep 1: Loading checkpoint data...")
    data_loader = Phase3DataLoader()
    
    try:
        checkpoint_data = data_loader.load_checkpoint_data()
        print("✓ Checkpoints loaded successfully")
    except Exception as e:
        print(f"❌ Failed to load checkpoints: {str(e)}")
        print("Please ensure you have run the previous sections and have the required checkpoints.")
        print("Required checkpoints:")
        print("  - data_loading.pkl (from Section 1)")
        print("  - feature_extraction.pkl (from Section 2)")
        print("  - data_splitting.pkl (from Section 3)")
        return None
    
    # Step 2: Prepare datasets
    print("\nStep 2: Preparing datasets...")
    
    try:
        dataset_info = data_loader.prepare_datasets(checkpoint_data)
        
        train_dataset = dataset_info['train_dataset']
        val_dataset = dataset_info['val_dataset']
        test_dataset = dataset_info['test_dataset']
        model_checkpoint = dataset_info['model_checkpoint']
        
        print(f"✓ Datasets prepared:")
        print(f"  Train: {len(train_dataset)} samples")
        print(f"  Val: {len(val_dataset)} samples")
        print(f"  Test: {len(test_dataset)} samples")
        
    except Exception as e:
        print(f"❌ Failed to prepare datasets: {str(e)}")
        import traceback
        traceback.print_exc()
        return None
    
    # Step 3: Initialize model and load weights
    print("\nStep 3: Initializing model...")
    model = OrthogonalEnsemble()
    
    # Load weights from checkpoint if available
    try:
        if model_checkpoint and 'model_state_dict' in model_checkpoint:
            model.load_state_dict(model_checkpoint['model_state_dict'])
            print("✓ Loaded model weights from checkpoint")
        else:
            print("⚠ No pre-trained weights found, starting from scratch")
    except Exception as e:
        print(f"⚠ Could not load model weights: {str(e)}")
        print("Starting with random initialization")
    
    # Step 4: Create data loaders
    print("\nStep 4: Creating data loaders...")
    train_loader = TorchDataLoader(
        train_dataset, 
        batch_size=Config.TEST_BATCH_SIZE if test_only else Config.FULL_BATCH_SIZE, 
        shuffle=True
    )
    val_loader = TorchDataLoader(
        val_dataset, 
        batch_size=Config.TEST_BATCH_SIZE if test_only else Config.FULL_BATCH_SIZE, 
        shuffle=False
    )
    test_loader = TorchDataLoader(
        test_dataset, 
        batch_size=Config.TEST_BATCH_SIZE if test_only else Config.FULL_BATCH_SIZE, 
        shuffle=False
    )
    
    print(f"✓ Data loaders created with batch size: {Config.TEST_BATCH_SIZE if test_only else Config.FULL_BATCH_SIZE}")
    
    # Step 5: Initialize trainer
    print("\nStep 5: Initializing trainer...")
    trainer = Phase3Trainer(model, Config.DEVICE)
    print(f"✓ Trainer initialized on {Config.DEVICE}")
    
    # Step 6: Run pipeline test
    print("\nStep 6: Running pipeline test...")
    test_success = trainer.run_pipeline_test(train_loader, val_loader)
    
    if not test_success:
        print("❌ Pipeline test failed. Please fix the errors before running full training.")
        return None
    
    print("✅ Pipeline test passed!")
    
    # Step 7: Run full training if requested
    if run_full:
        print("\nStep 7: Running full Phase 3 training...")
        
        # Update batch size for full training
        if test_only:
            train_loader = TorchDataLoader(train_dataset, batch_size=Config.FULL_BATCH_SIZE, shuffle=True)
            val_loader = TorchDataLoader(val_dataset, batch_size=Config.FULL_BATCH_SIZE, shuffle=False)
            test_loader = TorchDataLoader(test_dataset, batch_size=Config.FULL_BATCH_SIZE, shuffle=False)
        
        start_time = datetime.now()
        final_f1 = trainer.train_full(train_loader, val_loader, Config.FULL_EPOCHS)
        training_duration = (datetime.now() - start_time).total_seconds() / 60
        
        print(f"\nPhase 3 training completed in {training_duration:.1f} minutes")
        
        # Step 8: Evaluate on test set
        print("\nStep 8: Evaluating on test set...")
        evaluator = Evaluator(model, Config.DEVICE)
        test_results = evaluator.evaluate_on_test(test_loader)
        
        # Step 9: Save results
        print("\nStep 9: Saving results...")
        results_dir = os.path.join(Config.BASE_DIR, 'phase3_results')
        os.makedirs(results_dir, exist_ok=True)
        
        # Save model
        model_path = os.path.join(results_dir, 'phase3_best_model.pth')
        torch.save(model.state_dict(), model_path)
        print(f"✓ Model saved to {model_path}")
        
        # Save results
        results = {
            'final_f1': final_f1,
            'test_results': test_results,
            'training_results': trainer.results,
            'training_duration_minutes': training_duration,
            'config': {
                'epochs': Config.FULL_EPOCHS,
                'batch_size': Config.FULL_BATCH_SIZE,
                'learning_rates': Config.PHASE3_LR
            }
        }
        
        results_path = os.path.join(results_dir, 'phase3_results.pkl')
        with open(results_path, 'wb') as f:
            pickle.dump(results, f)
        print(f"✓ Results saved to {results_path}")
        
        # Print final summary
        print("\n" + "="*60)
        print("🎉 PHASE 3 TRAINING COMPLETE!")
        print("="*60)
        print(f"Final Validation F1: {final_f1:.4f}")
        print(f"Test F1: {test_results['final_metrics']['f1']:.4f}")
        print(f"Test AUC: {test_results['final_metrics']['auc']:.4f}")
        print(f"Training Duration: {training_duration:.1f} minutes")
        print(f"Results saved to: {results_dir}")
        
        return {
            'model': model,
            'results': results,
            'test_results': test_results
        }
    
    else:
        print("\nPipeline test completed successfully!")
        print("To run full training, call: run_phase3_pipeline(test_only=False, run_full=True)")
        return {'model': model, 'trainer': trainer}
        test_results = evaluator.evaluate_on_test(test_loader)
        
        # Step 9: Save results
        print("\nStep 9: Saving results...")
        results_dir = os.path.join(Config.BASE_DIR, 'phase3_results')
        os.makedirs(results_dir, exist_ok=True)
        
        # Save model
        model_path = os.path.join(results_dir, 'phase3_best_model.pth')
        torch.save(model.state_dict(), model_path)
        print(f"✓ Model saved to {model_path}")
        
        # Save results
        results = {
            'final_f1': final_f1,
            'test_results': test_results,
            'training_results': trainer.results,
            'training_duration_minutes': training_duration,
            'config': {
                'epochs': Config.FULL_EPOCHS,
                'batch_size': Config.FULL_BATCH_SIZE,
                'learning_rates': Config.PHASE3_LR
            }
        }
        
        results_path = os.path.join(results_dir, 'phase3_results.pkl')
        with open(results_path, 'wb') as f:
            pickle.dump(results, f)
        print(f"✓ Results saved to {results_path}")
        
        # Print final summary
        print("\n" + "="*60)
        print("🎉 PHASE 3 TRAINING COMPLETE!")
        print("="*60)
        print(f"Final Validation F1: {final_f1:.4f}")
        print(f"Test F1: {test_results['final_metrics']['f1']:.4f}")
        print(f"Test AUC: {test_results['final_metrics']['auc']:.4f}")
        print(f"Training Duration: {training_duration:.1f} minutes")
        print(f"Results saved to: {results_dir}")
        
        return {
            'model': model,
            'results': results,
            'test_results': test_results
        }

In [27]:
# ============================================================================
# USAGE EXAMPLES AND INSTRUCTIONS
# ============================================================================
def print_usage_instructions():
    """Print usage instructions"""
    print("=" * 80)
    print("📋 PHASE 3 PIPELINE USAGE INSTRUCTIONS")
    print("=" * 80)
    print()
    print("STEP 1: Verify checkpoints")
    print("  verify_checkpoints()")
    print()
    print("STEP 2: Run quick test (recommended)")
    print("  quick_test()")
    print()
    print("STEP 3: Run full training (after successful test)")
    print("  full_training()")
    print()
    print("ALTERNATIVE: Manual control")
    print("  # Test only")
    print("  result = run_phase3_pipeline(test_only=True, run_full=False)")
    print()
    print("  # Full training")
    print("  result = run_phase3_pipeline(test_only=False, run_full=True)")
    print()
    print("CONFIGURATION:")
    print(f"  Base directory: {Config.BASE_DIR}")
    print(f"  Device: {Config.DEVICE}")
    print(f"  Test batch size: {Config.TEST_BATCH_SIZE}")
    print(f"  Full batch size: {Config.FULL_BATCH_SIZE}")
    print(f"  Test epochs: {Config.TEST_EPOCHS}")
    print(f"  Full epochs: {Config.FULL_EPOCHS}")
    print()
    print("=" * 80)

# ============================================================================
# CHECKPOINT VERIFICATION UTILITY
# ============================================================================
def verify_checkpoints(base_dir=Config.BASE_DIR):
    """
    Utility function to verify available checkpoints
    """
    print("=" * 60)
    print("CHECKPOINT VERIFICATION")
    print("=" * 60)
    
    checkpoints_dir = os.path.join(base_dir, 'checkpoints')
    
    if not os.path.exists(checkpoints_dir):
        print(f"❌ Checkpoints directory does not exist: {checkpoints_dir}")
        return False
    
    required_checkpoints = [
        'data_loading.pkl',
        'feature_extraction.pkl', 
        'data_splitting.pkl'
    ]
    
    optional_checkpoints = [
        'orthogonal_ensemble.pkl',
        'ensemble_methods_proper.pkl'
    ]
    
    available_files = os.listdir(checkpoints_dir)
    available_checkpoints = [f for f in available_files if f.endswith('.pkl')]
    
    print(f"Checkpoints directory: {checkpoints_dir}")
    print(f"Available files: {len(available_files)}")
    print(f"Available checkpoints: {len(available_checkpoints)}")
    print()
    
    all_required_present = True
    
    print("REQUIRED CHECKPOINTS:")
    for checkpoint in required_checkpoints:
        if checkpoint in available_checkpoints:
            checkpoint_path = os.path.join(checkpoints_dir, checkpoint)
            size = os.path.getsize(checkpoint_path) / (1024*1024)  # MB
            print(f"  ✅ {checkpoint} ({size:.1f} MB)")
        else:
            print(f"  ❌ {checkpoint} - MISSING")
            all_required_present = False
    
    print("\nOPTIONAL CHECKPOINTS:")
    for checkpoint in optional_checkpoints:
        if checkpoint in available_checkpoints:
            checkpoint_path = os.path.join(checkpoints_dir, checkpoint)
            size = os.path.getsize(checkpoint_path) / (1024*1024)  # MB
            print(f"  ✅ {checkpoint} ({size:.1f} MB)")
        else:
            print(f"  ⚪ {checkpoint} - Not found")
    
    print("\nOTHER CHECKPOINTS:")
    other_checkpoints = [f for f in available_checkpoints 
                        if f not in required_checkpoints + optional_checkpoints]
    for checkpoint in other_checkpoints:
        checkpoint_path = os.path.join(checkpoints_dir, checkpoint)
        size = os.path.getsize(checkpoint_path) / (1024*1024)  # MB
        print(f"  ℹ️  {checkpoint} ({size:.1f} MB)")
    
    print()
    if all_required_present:
        print("✅ All required checkpoints are present!")
        print("You can proceed with Phase 3 training.")
    else:
        print("❌ Some required checkpoints are missing!")
        print("Please run the previous sections to generate the required checkpoints:")
        print("  - Section 1: Data Loading")
        print("  - Section 2: Feature Extraction") 
        print("  - Section 3: Data Splitting")
    
    return all_required_present

# ============================================================================
# QUICK START FUNCTIONS
# ============================================================================
def quick_test():
    """Quick test to verify pipeline works"""
    print("🚀 RUNNING QUICK PIPELINE TEST")
    print("This will verify all components work without full training")
    print()
    
    # Verify checkpoints first
    if not verify_checkpoints():
        return False
    
    # Run pipeline test
    result = run_phase3_pipeline(test_only=True, run_full=False)
    return result is not None

def full_training():
    """Run full Phase 3 training"""
    print("🚀 RUNNING FULL PHASE 3 TRAINING")
    print("This will run the complete end-to-end training")
    print()
    
    # Verify checkpoints first
    if not verify_checkpoints():
        return None
    
    # Ask for confirmation
    response = input("This will run full training which may take hours. Continue? (y/N): ")
    if response.lower() != 'y':
        print("Training cancelled.")
        return None
    
    # Run full training
    result = run_phase3_pipeline(test_only=False, run_full=True)
    return result



# ============================================================================
# USAGE EXAMPLES
# ============================================================================
if __name__ == "__main__":
    # Print usage instructions
    print_usage_instructions()
    
    # First verify checkpoints
    print("\nStep 1: Verifying checkpoints...")
    if verify_checkpoints():
        print("\n✅ Checkpoints verified! You can proceed with training.")
        print("\nNext steps:")
        print("1. Run quick_test() to verify pipeline")
        print("2. Run full_training() for complete training")
    else:
        print("\n❌ Please ensure you have the required checkpoints before proceeding.")

📋 PHASE 3 PIPELINE USAGE INSTRUCTIONS

STEP 1: Verify checkpoints
  verify_checkpoints()

STEP 2: Run quick test (recommended)
  quick_test()

STEP 3: Run full training (after successful test)
  full_training()

ALTERNATIVE: Manual control
  # Test only
  result = run_phase3_pipeline(test_only=True, run_full=False)

  # Full training
  result = run_phase3_pipeline(test_only=False, run_full=True)

CONFIGURATION:
  Base directory: results/exp_3
  Device: cuda
  Test batch size: 64
  Full batch size: 128
  Test epochs: 1
  Full epochs: 150


Step 1: Verifying checkpoints...
CHECKPOINT VERIFICATION
Checkpoints directory: results/exp_3\checkpoints
Available files: 19
Available checkpoints: 13

REQUIRED CHECKPOINTS:
  ✅ data_loading.pkl (10.6 MB)
  ✅ feature_extraction.pkl (4908.5 MB)
  ✅ data_splitting.pkl (1.2 MB)

OPTIONAL CHECKPOINTS:
  ⚪ orthogonal_ensemble.pkl - Not found
  ✅ ensemble_methods_proper.pkl (2.5 MB)

OTHER CHECKPOINTS:
  ℹ️  advanced_ensemble_methods.pkl (0.4 MB)
  ℹ️  com

In [22]:
quick_test()

🚀 RUNNING QUICK PIPELINE TEST
This will verify all components work without full training

CHECKPOINT VERIFICATION
Checkpoints directory: results/exp_3\checkpoints
Available files: 19
Available checkpoints: 13

REQUIRED CHECKPOINTS:
  ✅ data_loading.pkl (10.6 MB)
  ✅ feature_extraction.pkl (4908.5 MB)
  ✅ data_splitting.pkl (1.2 MB)

OPTIONAL CHECKPOINTS:
  ⚪ orthogonal_ensemble.pkl - Not found
  ✅ ensemble_methods_proper.pkl (2.5 MB)

OTHER CHECKPOINTS:
  ℹ️  advanced_ensemble_methods.pkl (0.4 MB)
  ℹ️  complete_diversity_ensemble_methods.pkl (0.4 MB)
  ℹ️  comprehensive_ensemble_tuning_timed.pkl (1.0 MB)
  ℹ️  ensemble_methods_updated.pkl (1.2 MB)
  ℹ️  error_analysis.pkl (3.2 MB)
  ℹ️  hyperparameter_tuning_ensemble_optimized.pkl (3.9 MB)
  ℹ️  ml_models_enhanced.pkl (25.8 MB)
  ℹ️  streamlined_ensemble_tuning.pkl (0.4 MB)
  ℹ️  tabnet_model.pkl (23.4 MB)

✅ All required checkpoints are present!
You can proceed with Phase 3 training.
🚀 PHASE 3 END-TO-END TRAINING PIPELINE
Device: cud

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Tokenized 62120 sequence windows
✓ Datasets prepared:
  Train: 36419 samples
  Val: 6426 samples
  Test: 10122 samples

Step 3: Initializing model...


NameError: name 'ImprovedFusionNetwork' is not defined

In [29]:
full_training()

🚀 RUNNING FULL PHASE 3 TRAINING
This will run the complete end-to-end training

CHECKPOINT VERIFICATION
Checkpoints directory: results/exp_3\checkpoints
Available files: 19
Available checkpoints: 13

REQUIRED CHECKPOINTS:
  ✅ data_loading.pkl (10.6 MB)
  ✅ feature_extraction.pkl (4908.5 MB)
  ✅ data_splitting.pkl (1.2 MB)

OPTIONAL CHECKPOINTS:
  ⚪ orthogonal_ensemble.pkl - Not found
  ✅ ensemble_methods_proper.pkl (2.5 MB)

OTHER CHECKPOINTS:
  ℹ️  advanced_ensemble_methods.pkl (0.4 MB)
  ℹ️  complete_diversity_ensemble_methods.pkl (0.4 MB)
  ℹ️  comprehensive_ensemble_tuning_timed.pkl (1.0 MB)
  ℹ️  ensemble_methods_updated.pkl (1.2 MB)
  ℹ️  error_analysis.pkl (3.2 MB)
  ℹ️  hyperparameter_tuning_ensemble_optimized.pkl (3.9 MB)
  ℹ️  ml_models_enhanced.pkl (25.8 MB)
  ℹ️  streamlined_ensemble_tuning.pkl (0.4 MB)
  ℹ️  tabnet_model.pkl (23.4 MB)

✅ All required checkpoints are present!
You can proceed with Phase 3 training.


This will run full training which may take hours. Continue? (y/N):  y


🚀 PHASE 3 END-TO-END TRAINING PIPELINE
Device: cuda

Step 1: Loading checkpoint data...
Loading data from checkpoints...
Available checkpoints: ['advanced_ensemble_methods.pkl', 'complete_diversity_ensemble_methods.pkl', 'comprehensive_ensemble_tuning_timed.pkl', 'data_loading.pkl', 'data_splitting.pkl', 'ensemble_methods_proper.pkl', 'ensemble_methods_updated.pkl', 'error_analysis.pkl', 'feature_extraction.pkl', 'hyperparameter_tuning_ensemble_optimized.pkl', 'ml_models_enhanced.pkl', 'streamlined_ensemble_tuning.pkl', 'tabnet_model.pkl']
✓ Loaded data_loading checkpoint
✓ Loaded feature_extraction checkpoint
✓ Loaded data_splitting checkpoint
Found existing model checkpoints: ['advanced_ensemble_methods.pkl', 'complete_diversity_ensemble_methods.pkl', 'comprehensive_ensemble_tuning_timed.pkl', 'ensemble_methods_proper.pkl', 'ensemble_methods_updated.pkl', 'hyperparameter_tuning_ensemble_optimized.pkl', 'streamlined_ensemble_tuning.pkl']
✓ Loaded model checkpoint: streamlined_ensemble

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Tokenized 62120 sequence windows
✓ Datasets prepared:
  Train: 36419 samples
  Val: 6426 samples
  Test: 10122 samples

Step 3: Initializing model...
⚠ No pre-trained weights found, starting from scratch

Step 4: Creating data loaders...
✓ Data loaders created with batch size: 128

Step 5: Initializing trainer...
✓ Trainer initialized on cuda

Step 6: Running pipeline test...

RUNNING PIPELINE TEST (1 EPOCH)


NameError: name 'ImprovedMultiObjectiveLoss' is not defined